# FDA RadiologyAI 7-Pillar Audit — Analysis Notebook 
**Target**: JACR Focus Issue 'AI and Data Science on the Global Stage' (January 2027)


## 0. Imports & setup

In [68]:
import os
import sys

projet_path = "/Users/anne-gabrielle/Le_Neuro/JACR_Article"

if projet_path not in sys.path:
    sys.path.append(projet_path)

In [69]:
import os, math, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from scipy import stats as scipy_stats
from scipy.stats import chi2_contingency
import statsmodels.formula.api as smf

from src.data_cleaning import (
    extract_nb_manufacturers, extract_manufacturers_list,
    extract_multi_protocol, extract_param_disclosure,
    extract_sample_size, extract_countries_c1d,
    extract_countries_d3,
    is_public_dataset, extract_hq_country, extract_validation_types,
    extract_predicate_year, extract_predicate_number,
    find_ancestor_chain, extract_h3_subflag,
    clean_comparator_method
)
from src.stats_metrics import (
    wilson_ci, reporting_rate_with_ci,
    herfindahl_hirschman_index, shannon_diversity,
    calculate_gini, lorenz_curve, cochran_armitage_trend_test,
)
from src.models import fit_logistic_regression, fit_ordinal_regression
from src.plotting import plot_reporting_cliff, plot_lorenz_curve

warnings.filterwarnings('ignore')
OUT_SUPP = '../output/supplementary'
OUT_MAIN = '../output/main_exhibits'
os.makedirs(OUT_SUPP, exist_ok=True)
os.makedirs(OUT_MAIN, exist_ok=True)
print('Setup complete.')


Setup complete.


## 1. Data loading

In [70]:
df = pd.read_excel('../data/FDA_Clean_Dataset.xlsx')
print(f'Loaded {df.shape[0]:,} devices x {df.shape[1]} columns')
df.head(3)


Loaded 1,089 devices x 56 columns


,A1-Submission_Number,Published Year,A2-Device_and_Company,A3-Decision_Date,A4-Clearance_Info,Pathway,B1-Body_Region_and_Modality,Primary_Body_Region,Primary_Modality,B2-Clinical_Task_and_Role,...,J1_Score,J2-Demographic_Reporting_Score,J2_Score,J3-Scanner_Diversity_Score,J3_Score,J4-Overall_Generalizability_Risk,J4_Composite_Score,J4_Risk_Level,J5-Key_Generalizability_Gaps,K1-Document_Quality
0,K251169,2025.0,Vivid Pioneer; GE Medical Systems Ultrasound a...,NR,510(k); NR; NR,510(k),Cardiovascular; Ultrasound,Cardiovascular,Ultrasound,Quantification/Measurement; Decision support/a...,...,0,0 – No demographics reported,0,3 – 2 manufacturers,3,Score: 3; Risk: High risk,3,High risk,- Limited subgroup analysis beyond BMI groups;...,Rich
1,K251408,2025.0,OsteoSight™ Hip (v1); Naitive Technologies Ltd,09/02/2025,510(k); SAO; 820.1171,510(k),Abdomen/Pelvis; X-ray,Abdomen/Pelvis,X-ray,Quantification/Measurement; Decision support/a...,...,2,5 – Full demographics + subgroup performance a...,5,0 – Not reported,0,Score: 7; Risk: Moderate risk,7,Moderate risk,"- Multi-site validation but limited to the US,...",Rich
2,K253091,2025.0,ART-Plan+; TheraPanacea SAS,12/23/2025,510(k); MUJ; 892.5050,510(k),- Body region: Head/Brain; Chest/Thorax; Abdom...,Head/Brain,MRI,Segmentation; Stand-alone diagnostic │ Treatme...,...,1,0 – No demographics reported,0,0 – Not reported,0,Score: 1; Risk: High risk,1,High risk,1. Single-site US validation only: The device ...,Sparse


## 2. Master cleaning block
All derived columns computed once and reused by every pillar.


In [71]:
df = df.rename(columns={'Published Year': 'Published_Year'})

df['Clinical_Task'] = df['B2-Clinical_Task_and_Role'].apply(
    lambda x: str(x).split(';')[0].strip() if pd.notnull(x) and ';' in str(x) else 'NR')
df['HQ_Country'] = df['B4-Company_Country_and_Market'].apply(extract_hq_country)

df['Train_Countries_Set'] = df['C1d-Training_Data_Countries'].apply(extract_countries_c1d)
df['Val_Countries_Set']   = df['D3-Validation_Sites_Detail'].apply(extract_countries_d3)
df['Val_Types_Clean']     = df['D3-Validation_Sites_Detail'].apply(extract_validation_types)

# World Bank / region maps (used by P4 & P5)
WORLD_BANK = {
    'USA':'High-Income','UNITED STATES':'High-Income','GERMANY':'High-Income',
    'FRANCE':'High-Income','ISRAEL':'High-Income','SOUTH KOREA':'High-Income',
    'KOREA':'High-Income','JAPAN':'High-Income','UNITED KINGDOM':'High-Income',
    'UK':'High-Income','NETHERLANDS':'High-Income','SWITZERLAND':'High-Income',
    'CANADA':'High-Income','AUSTRALIA':'High-Income','SWEDEN':'High-Income',
    'DENMARK':'High-Income','CHINA':'LMIC','INDIA':'LMIC','BRAZIL':'LMIC',
    'SOUTH AFRICA':'LMIC','TURKEY':'LMIC','MEXICO':'LMIC',
    'ARGENTINA':'LMIC','INDONESIA':'LMIC',
}
REGION_MAP = {
    'USA':'North America','UNITED STATES':'North America','CANADA':'North America',
    'GERMANY':'Europe','FRANCE':'Europe','UNITED KINGDOM':'Europe','UK':'Europe',
    'NETHERLANDS':'Europe','SWITZERLAND':'Europe','SWEDEN':'Europe','DENMARK':'Europe',
    'ISRAEL':'Middle East',
    'SOUTH KOREA':'East Asia','KOREA':'East Asia','JAPAN':'East Asia','CHINA':'East Asia',
    'INDIA':'South Asia',
}
df['Income_Bracket'] = df['HQ_Country'].map(WORLD_BANK).fillna('Unclassified')
df['HQ_Region']      = df['HQ_Country'].map(REGION_MAP).fillna('Other')

total_devices = len(df)
print(f'Cohort: {total_devices:,} devices')
print(df['Pathway'].value_counts())


Cohort: 1,089 devices
Pathway
510(k)     1071
PMA           8
De Novo       7
NR            3
Name: count, dtype: int64


## 3. Shared predicate map (Pillars 3 & 6)

In [72]:
predicate_map = dict(zip(
    df['A1-Submission_Number'].astype(str).str.strip(),
    df['H1-Predicate_Chain'].apply(extract_predicate_number)
))
df['Predicate_Chain_Depth'] = df['A1-Submission_Number'].apply(
    lambda sid: len(find_ancestor_chain(predicate_map, sid)))
predicate_years     = df['H1-Predicate_Chain'].apply(extract_predicate_year)
df['Predicate_Age'] = df['Published_Year'] - predicate_years
print('Predicate map built.')
print(f"  With predicate: {df['Predicate_Chain_Depth'].gt(0).sum():,}")
print(f"  Mean age: {df['Predicate_Age'].mean():.1f} yrs")


Predicate map built.
  With predicate: 854
  Mean age: -3.1 yrs


## 4. Shared helpers

In [73]:
def _reported(val):
    s = str(val).strip()
    return (pd.notnull(val)
            and s.upper() not in {'NR','NAN',''}
            and not s.lower().startswith('not reported')
            and not s.lower().startswith('no '))

DEMO_FIELDS = {
    'Training Age':  'C2b-Training_Age_Distribution',
    'Training Sex':  'C2c-Training_Sex_Distribution',
    'Training Race': 'C2d-Training_Race_Ethnicity',
    'Val Age':        'E1-Val_Age',
    'Val Sex':        'E2-Val_Sex',
    'Val Race':       'E3-Val_Race',
    'Val BMI':        'E4-Val_BMI',
    'Val Prevalence': 'E5-Val_Disease_Prevalence',
}
print('Helpers ready.')


Helpers ready.


---
## Pillar 1 — The Paradox of Volume vs. Value (Brittleness Gap)


### 1.1 Reporting rates with 95% Wilson CIs

In [74]:
reporting_rows, reporting_rates_for_fig, ci_dict_for_fig = [], {}, {}
for label, col in DEMO_FIELDS.items():
    if col not in df.columns: continue
    res = reporting_rate_with_ci(df[col], _reported)
    reporting_rows.append({'Field': label, 'Column': col, **res})
    reporting_rates_for_fig[label] = res['rate']
    ci_dict_for_fig[label] = (res['ci_low'], res['ci_high'])

td_desc = reporting_rate_with_ci(df['C1a-Training_Data_Described'],
                                  lambda x: str(x).strip().upper() == 'YES')
td_demo = reporting_rate_with_ci(df['C2a-Training_Demographics_Reported'],
                                  lambda x: str(x).strip().upper() == 'YES')

df_p1_rates = pd.DataFrame(reporting_rows)
df_p1_rates.to_excel(f'{OUT_SUPP}/P1-reporting_rates.xlsx', index=False)
print(f"Training-data description rate: {td_desc['rate']:.1%} [{td_desc['ci_low']:.1%}-{td_desc['ci_high']:.1%}]")
print(f"Training-demographic rate:      {td_demo['rate']:.1%} [{td_demo['ci_low']:.1%}-{td_demo['ci_high']:.1%}]")
df_p1_rates


Training-data description rate: 44.3% [41.3%-47.2%]
Training-demographic rate:      28.4% [25.8%-31.1%]


,Field,Column,rate,ci_low,ci_high,n,k
0,Training Age,C2b-Training_Age_Distribution,0.247934,0.223203,0.274437,1089,270
1,Training Sex,C2c-Training_Sex_Distribution,0.207530,0.184492,0.232624,1089,226
2,Training Race,C2d-Training_Race_Ethnicity,0.056933,0.044664,0.072317,1089,62
3,Val Age,E1-Val_Age,0.233242,0.209089,0.259270,1089,254
4,Val Sex,E2-Val_Sex,0.271809,0.246222,0.299001,1089,296
5,Val Race,E3-Val_Race,0.101928,0.085335,0.121320,1089,111
6,Val BMI,E4-Val_BMI,0.083563,0.068554,0.101499,1089,91
7,Val Prevalence,E5-Val_Disease_Prevalence,0.112948,0.095492,0.133124,1089,123


### 1.2 Cliff, HHI, composite J, J4 risk, K1 quality

In [75]:
r_age  = reporting_rates_for_fig.get('Val Age', float('nan'))
r_race = reporting_rates_for_fig.get('Val Race', float('nan'))
cliff  = r_age - r_race
print(f'Demographic cliff (Age - Race): {cliff:.1%}')

flat_tc = [c for lst in df['Train_Countries_Set'] for c in lst]
hhi_training = herfindahl_hirschman_index(pd.Series(flat_tc).value_counts().values)
print(f'Training-country HHI: {hhi_training:.0f}')

valid_j = df['J4_Composite_Score'].dropna()
print(f'Mean composite J: {valid_j.mean():.2f} +/- {valid_j.std():.2f} SD')

risk_clean = df['J4_Risk_Level'].astype(str).str.strip()
for cat_label, data_label in [('High risk (0-5)', 'High risk'), 
                              ('Moderate risk (6-10)', 'Moderate risk'), 
                              ('Low risk (11-15)', 'Low risk')]:
    nk = int((risk_clean == data_label).sum())
    lo_ci, hi_ci = wilson_ci(nk, total_devices)
    print(f'  {cat_label}: {nk/total_devices:.1%} [{lo_ci:.1%}-{hi_ci:.1%}]')

j4_by_pathway = (df.groupby('Pathway')['J4_Composite_Score']
                   .agg(['mean','median','std','count'])
                   .rename(columns={'mean':'Mean J4','median':'Median J4','std':'SD','count':'N'}))
j4_by_pathway.to_excel(f'{OUT_SUPP}/P1_J4_by_pathway.xlsx')

print('\nK1 document quality:')
print(df['K1-Document_Quality'].value_counts(normalize=True).map('{:.1%}'.format))


Demographic cliff (Age - Race): 13.1%
Training-country HHI: 2964
Mean composite J: 2.97 +/- 3.14 SD
  High risk (0-5): 78.1% [75.5%-80.4%]
  Moderate risk (6-10): 19.8% [17.6%-22.3%]
  Low risk (11-15): 2.1% [1.4%-3.1%]

K1 document quality:
K1-Document_Quality
Sparse      35.7%
Moderate    32.0%
Rich        18.4%
Opaque      14.0%
Name: proportion, dtype: str


### 1.3 Top recurring gaps from J5 free-text

In [76]:
if 'J5-Key_Generalizability_Gaps' in df.columns:
    gap_kw = [
        'single-site','single site','single country','no race','no ethnicity',
        'single scanner','single manufacturer','training not disclosed',
        'no subgroup','us only','usa only','academic only'
    ]
    gap_freq = {kw: df['J5-Key_Generalizability_Gaps'].apply(
        lambda x: kw.lower() in str(x).lower()).sum() for kw in gap_kw}
    df_j5 = pd.DataFrame(sorted(gap_freq.items(), key=lambda x: -x[1]),
                          columns=['Gap theme','Count'])
    df_j5['pct'] = (df_j5['Count']/total_devices*100).round(1)
    df_j5.to_excel(f'{OUT_SUPP}/P1_J5_gap_themes.xlsx', index=False)
    print(df_j5.head(10).to_string(index=False))
else:
    print('J5 column not found - add J5-Key_Generalizability_Gaps to extraction.')


             Gap theme  Count  pct
           single-site    384 35.3
               no race    338 31.0
           no subgroup    327 30.0
        single scanner    230 21.1
           single site     20  1.8
               us only      5  0.5
   single manufacturer      3  0.3
        single country      1  0.1
          no ethnicity      1  0.1
training not disclosed      0  0.0


### 1.4 Chi-square / Fisher's exact — reporting across pathways (Bonferroni)

In [77]:
alpha_bonf = 0.05 / len(DEMO_FIELDS)
chi2_rows = []
for label, col in DEMO_FIELDS.items():
    if col not in df.columns: continue
    reported = df[col].apply(_reported).astype(int)
    ct = pd.crosstab(df['Pathway'], reported)
    if ct.shape[1] == 2 and len(ct) >= 2:
        stat, p, dof, _ = chi2_contingency(ct)
        chi2_rows.append({'Field':label,'chi2':round(stat,3),'p':round(p,4),
                          'sig_bonf': p < alpha_bonf})
df_chi2 = pd.DataFrame(chi2_rows)
df_chi2.to_excel(f'{OUT_SUPP}/P1_chi2_pathway_reporting.xlsx', index=False)
print(f'Bonferroni alpha = {alpha_bonf:.5f}')
print(df_chi2.to_string(index=False))


Bonferroni alpha = 0.00625
         Field   chi2      p  sig_bonf
  Training Age  2.216 0.5289     False
  Training Sex  1.054 0.7881     False
 Training Race  1.627 0.6534     False
       Val Age  1.033 0.7932     False
       Val Sex  1.744 0.6272     False
      Val Race  3.832 0.2802     False
       Val BMI  4.741 0.1917     False
Val Prevalence 28.555 0.0000      True


### 1.5 Cochran-Armitage trend test (year-over-year)

In [78]:
for label, col in DEMO_FIELDS.items():
    if col not in df.columns: continue
    flag = df[col].apply(_reported).astype(int)
    r = cochran_armitage_trend_test(df['Published_Year'], flag)
    sig = '(*)' if r['p_value'] < 0.05 else '(ns)'
    print(f"{label:25s}  T={r['T_statistic']:+.3f}  p={r['p_value']:.4f}  {sig}")


Training Age               T=+8.558  p=0.0000  (*)
Training Sex               T=+7.417  p=0.0000  (*)
Training Race              T=+4.625  p=0.0000  (*)
Val Age                    T=+7.913  p=0.0000  (*)
Val Sex                    T=+7.705  p=0.0000  (*)
Val Race                   T=+6.535  p=0.0000  (*)
Val BMI                    T=+4.906  p=0.0000  (*)
Val Prevalence             T=-0.046  p=0.9631  (ns)


### 1.6 Logistic regression — any demographic reporting

In [79]:
df['Any_Demo_Reported'] = (
    df['C2a-Training_Demographics_Reported'].astype(str).str.upper() == 'YES'
).astype(int)

valid_modalities = ['CT', 'MRI', 'Ultrasound', 'X-ray', 'Mammography', 'PET/CT']
df['Modality_Clean'] = df['Primary_Modality'].apply(lambda x: x if x in valid_modalities else 'Other')

invalid_regions = ['Head/neck', 'Ophthalmology', 'Body']
df['Body_Region_Clean'] = df['Primary_Body_Region'].apply(lambda x: 'Other' if x in invalid_regions else x)

logit_p1 = fit_logistic_regression(
    df, 'Any_Demo_Reported',
    ['Published_Year', 'Modality_Clean', 'Body_Region_Clean']
)
print(logit_p1.summary())


                           Logit Regression Results                           
Dep. Variable:      Any_Demo_Reported   No. Observations:                 1088
Model:                          Logit   Df Residuals:                     1071
Method:                           MLE   Df Model:                           16
Date:                Tue, 26 May 2026   Pseudo R-squ.:                  0.1477
Time:                        16:51:14   Log-Likelihood:                -552.53
converged:                       True   LL-Null:                       -648.28
Covariance Type:            nonrobust   LLR p-value:                 4.101e-32
                                             coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------
Intercept                               -786.5020     86.790     -9.062      0.000    -956.608    -616.396
C(Modality_Clean)[T.MRI]                  -0.09

### 1.7 Ordinal logistic regression — K1 quality (inter-pathway gradient)

In [80]:
ord_k1 = fit_ordinal_regression(
    df, 'K1-Document_Quality',
    ['Opaque','Sparse','Moderate','Rich'],
    ['Pathway','Published_Year','Primary_Modality'])
print(ord_k1.summary())


                              OrderedModel Results                             
Dep. Variable:     K1-Document_Quality   Log-Likelihood:                -1329.1
Model:                    OrderedModel   AIC:                             2694.
Method:             Maximum Likelihood   BIC:                             2784.
Date:                 Tue, 26 May 2026                                         
Time:                         16:51:15                                         
No. Observations:                 1088                                         
Df Residuals:                     1070                                         
Df Model:                           15                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Pathway_De Novo                  0.5288      0.662      0.799      0.424      -0.768  

### 1.8 Stratified aggregation (one file per stratification variable)

In [81]:
def _compute_group_rates(group):
    total = len(group)
    if total == 0: return pd.Series(dtype=float)
    row = {'N': total}
    row['Training_desc_rate'] = (
        group['C1a-Training_Data_Described'].astype(str).str.upper()=='YES').sum()/total
    row['Training_demo_rate'] = (
        group['C2a-Training_Demographics_Reported'].astype(str).str.upper()=='YES').sum()/total
    for lbl, col in DEMO_FIELDS.items():
        if col in group.columns:
            k = int(group[col].apply(_reported).sum())
            row[f'Rate_{lbl}'] = k/total
            row[f'CI_lo_{lbl}'],row[f'CI_hi_{lbl}'] = wilson_ci(k,total)
    return pd.Series(row)

for strat in ['Pathway','Published_Year','Primary_Modality','Clinical_Task']:
    df.groupby(strat).apply(_compute_group_rates).reset_index().to_excel(
        f'{OUT_SUPP}/P1_stratified_by_{strat}.xlsx', index=False)
print('Stratified files saved.')


Stratified files saved.


### 1.9 Table 1 — Summary cohort table with 95% CIs

In [82]:
rows_t1 = []
for grp in ['Overall'] + sorted(df['Pathway'].dropna().unique().tolist()):
    sub = df if grp=='Overall' else df[df['Pathway']==grp]
    n = len(sub)
    row = {'Group':grp, 'N':n}
    for lbl, col in DEMO_FIELDS.items():
        if col not in sub.columns: continue
        k = int(sub[col].apply(_reported).sum())
        lo,hi = wilson_ci(k,n)
        row[f'{lbl} rate'] = f'{k/n:.1%}'
        row[f'{lbl} 95%CI'] = f'[{lo:.1%}-{hi:.1%}]'
    rows_t1.append(row)
df_table1 = pd.DataFrame(rows_t1)
df_table1.to_excel(f'{OUT_SUPP}/Table1_cohort_summary.xlsx', index=False)
print('Table 1 saved.'); df_table1


Table 1 saved.


,Group,N,Training Age rate,Training Age 95%CI,Training Sex rate,Training Sex 95%CI,Training Race rate,Training Race 95%CI,Val Age rate,Val Age 95%CI,Val Sex rate,Val Sex 95%CI,Val Race rate,Val Race 95%CI,Val BMI rate,Val BMI 95%CI,Val Prevalence rate,Val Prevalence 95%CI
0,Overall,1089,24.8%,[22.3%-27.4%],20.8%,[18.4%-23.3%],5.7%,[4.5%-7.2%],23.3%,[20.9%-25.9%],27.2%,[24.6%-29.9%],10.2%,[8.5%-12.1%],8.4%,[6.9%-10.1%],11.3%,[9.5%-13.3%]
1,510(k),1071,24.7%,[22.3%-27.4%],20.8%,[18.5%-23.4%],5.7%,[4.5%-7.2%],23.3%,[20.9%-26.0%],27.4%,[24.8%-30.1%],10.2%,[8.5%-12.1%],8.3%,[6.8%-10.1%],10.6%,[8.9%-12.6%]
2,De Novo,7,42.9%,[15.8%-75.0%],14.3%,[2.6%-51.3%],14.3%,[2.6%-51.3%],28.6%,[8.2%-64.1%],14.3%,[2.6%-51.3%],28.6%,[8.2%-64.1%],28.6%,[8.2%-64.1%],57.1%,[25.0%-84.2%]
3,NR,3,0.0%,[0.0%-56.2%],0.0%,[0.0%-56.2%],0.0%,[0.0%-56.2%],0.0%,[0.0%-56.2%],0.0%,[0.0%-56.2%],0.0%,[0.0%-56.2%],0.0%,[0.0%-56.2%],33.3%,[6.1%-79.2%]
4,PMA,8,25.0%,[7.1%-59.1%],25.0%,[7.1%-59.1%],0.0%,[0.0%-32.4%],25.0%,[7.1%-59.1%],25.0%,[7.1%-59.1%],0.0%,[0.0%-32.4%],0.0%,[0.0%-32.4%],50.0%,[21.5%-78.5%]


### Fig 1A — Reporting cliff

In [83]:
fig1a = plot_reporting_cliff(
    reporting_rates_for_fig, ci_dict=ci_dict_for_fig,
    title=f'Demographic reporting cliff (N={total_devices:,})',
    save_path=f'{OUT_MAIN}/Fig1A_reporting_cliff.png')
plt.show(); print('Fig 1A saved.')


Fig 1A saved.


### Fig 1B — J4 risk by pathway (stacked bar)

In [84]:
risk_order = ['High risk','Moderate risk','Low risk']
risk_colors = ['#c0392b','#e67e22','#2980b9']
fig1b, axes = plt.subplots(1, 2, figsize=(12,5), gridspec_kw={'width_ratios':[1,3]})
overall_counts = df['J4_Risk_Level'].value_counts()
vals_ov = [overall_counts.get(r,0)/total_devices for r in risk_order]
bottom = 0
for val, col, lbl in zip(vals_ov, risk_colors, risk_order):
    axes[0].bar(['Overall'], val, bottom=bottom, color=col, label=lbl)
    if val > 0.02:
        axes[0].text(0, bottom+val/2, f'{val:.0%}', ha='center', va='center',
                     fontsize=9, color='white', fontweight='bold')
    bottom += val
axes[0].set_ylim(0,1); axes[0].set_ylabel('Proportion'); axes[0].set_title('Overall')
axes[0].legend(loc='lower right', fontsize=7)
pathway_risk = (df.groupby('Pathway')['J4_Risk_Level']
                  .value_counts(normalize=True).unstack(fill_value=0)
                  .reindex(columns=[r for r in risk_order
                                     if r in df['J4_Risk_Level'].dropna().unique()]))
bottom2 = np.zeros(len(pathway_risk))
for col_r, color_r in zip(pathway_risk.columns, risk_colors):
    axes[1].bar(pathway_risk.index, pathway_risk[col_r], bottom=bottom2, color=color_r, label=col_r)
    bottom2 += pathway_risk[col_r].values
axes[1].set_ylim(0,1); axes[1].set_title('By regulatory pathway')
axes[1].tick_params(axis='x', rotation=20); axes[1].legend(loc='lower right', fontsize=7)
plt.suptitle('J4 generalizability risk distribution', fontsize=12, fontweight='bold')
plt.tight_layout()
fig1b.savefig(f'{OUT_MAIN}/Fig1B_J4_risk_by_pathway.png', dpi=150, bbox_inches='tight')
plt.show(); print('Fig 1B saved.')


Fig 1B saved.


### Fig 1C — Heatmap: Year x Reporting field

In [85]:
years_all = sorted(df['Published_Year'].dropna().unique())
hmap = {}
for lbl, col in DEMO_FIELDS.items():
    if col not in df.columns: continue
    hmap[lbl] = {yr: df[df['Published_Year']==yr][col].apply(_reported).mean()
                 for yr in years_all}
df_hmap = pd.DataFrame(hmap, index=years_all)
df_hmap.to_excel(f'{OUT_SUPP}/P1_heatmap_data.xlsx')
fig1c, ax1c = plt.subplots(figsize=(max(8,len(DEMO_FIELDS)*1.2), max(4,len(years_all)*0.4)))
im = ax1c.imshow(df_hmap.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
ax1c.set_xticks(range(len(df_hmap.columns)))
ax1c.set_xticklabels(df_hmap.columns, rotation=40, ha='right', fontsize=8)
ax1c.set_yticks(range(len(years_all))); ax1c.set_yticklabels(years_all, fontsize=8)
for i in range(len(years_all)):
    for j in range(len(df_hmap.columns)):
        val = df_hmap.values[i,j]
        if not (val != val):  # not nan
            ax1c.text(j, i, f'{val:.0%}', ha='center', va='center', fontsize=7,
                      color='black' if val < 0.6 else 'white')
plt.colorbar(im, ax=ax1c, label='% reporting')
ax1c.set_title('Reporting rate by year and field')
plt.tight_layout()
fig1c.savefig(f'{OUT_SUPP}/Fig1C_heatmap_year_field.png', dpi=150, bbox_inches='tight')
plt.show(); print('Fig 1C saved.')


Fig 1C saved.


### Fig 1D — Sankey data export (Pathway -> Demo -> J4 risk)

In [86]:
df['Demo_Flag'] = (df['C2a-Training_Demographics_Reported'].astype(str).str.upper()=='YES'
                  ).map({True:'Demo: Yes', False:'Demo: No'})
df_sankey = df[df['J4_Risk_Level'].notna()][['Pathway','Demo_Flag','J4_Risk_Level']].copy()
df_sankey.to_excel(f'{OUT_SUPP}/P1_Fig1D_sankey_data.xlsx', index=False)
print('Sankey data saved. Use plotly.graph_objects Sankey for final figure.')
try:
    import plotly.graph_objects as go
    all_nodes = (list(df_sankey['Pathway'].unique())+list(df_sankey['Demo_Flag'].unique())
                 +list(df_sankey['J4_Risk_Level'].unique()))
    nidx = {n:i for i,n in enumerate(all_nodes)}
    src,tgt,vals=[],[],[]
    for (p,d),c in df_sankey.groupby(['Pathway','Demo_Flag']).size().items():
        src.append(nidx[p]); tgt.append(nidx[d]); vals.append(c)
    for (d,j),c in df_sankey.groupby(['Demo_Flag','J4_Risk_Level']).size().items():
        src.append(nidx[d]); tgt.append(nidx[j]); vals.append(c)
    fig1d = go.Figure(go.Sankey(
        node=dict(label=all_nodes,pad=15,thickness=20),
        link=dict(source=src,target=tgt,value=vals)))
    fig1d.update_layout(title='Pathway -> Demo reported -> J4 risk')
    fig1d.write_image(f'{OUT_SUPP}/Fig1D_sankey.png')
    print('Fig 1D saved (plotly).')
except ImportError:
    print('plotly not installed: pip install plotly kaleido')


Sankey data saved. Use plotly.graph_objects Sankey for final figure.
Fig 1D saved (plotly).


---
## Pillar 2 — Radiological 'World Models' (Scanner Physics)


### 2.1 Aggregation metrics

In [87]:
stats_p2 = {}
nb_mfr = df['F1-Scanner_Manufacturers'].apply(extract_nb_manufacturers)
stats_p2['Scanner-reporting rate']           = df['F1-Scanner_Manufacturers'].apply(
    lambda x: pd.notnull(x) and str(x).strip().upper() not in {'NR','NOT REPORTED'}
).sum()/total_devices
stats_p2['Mean # manufacturers']             = nb_mfr.mean()
stats_p2['Median # manufacturers']           = nb_mfr.median()
stats_p2['IQR # manufacturers']              = nb_mfr.quantile(0.75)-nb_mfr.quantile(0.25)
stats_p2['Min # manufacturers']              = nb_mfr.min()
stats_p2['Max # manufacturers']              = nb_mfr.max()
stats_p2['Multi-vendor rate (J3>=3)']        = (df['J3_Score']>=3).sum()/total_devices
stats_p2['Multi-protocol testing rate']      = df['F3-Acquisition_and_Protocols'].apply(
    extract_multi_protocol).sum()/total_devices
stats_p2['Param disclosure rate']            = df['F3-Acquisition_and_Protocols'].apply(
    extract_param_disclosure).sum()/total_devices
stats_p2['Architecture transparency rate']   = df['I1-AI_ML_Method'].apply(
    lambda x: pd.notnull(x) and str(x).strip().upper() not in
    {'NR','PROPRIETARY','PROPRIETARY (NOT DISCLOSED)'}).sum()/total_devices

def _phys(row):
    i1=str(row.get('I1-AI_ML_Method','')).upper()
    i2=str(row.get('I2-Input_and_Output','')).upper()
    return any(t in i1 or t in i2 for t in ['PINN','PHYSICS','RAW DATA','RAW-DATA'])
stats_p2['Physics-aware architecture rate']  = df.apply(_phys,axis=1).sum()/total_devices

all_mfr=[m for lst in df['F1-Scanner_Manufacturers'].apply(extract_manufacturers_list) for m in lst]
if all_mfr:
    mfr_counts=pd.Series(all_mfr).value_counts()
    stats_p2['Manufacturer HHI']              = herfindahl_hirschman_index(mfr_counts.values)
    stats_p2['Manufacturer Shannon diversity'] = shannon_diversity(mfr_counts.values)
else:
    stats_p2['Manufacturer HHI']=stats_p2['Manufacturer Shannon diversity']=float('nan')

crosstab_mod_j3  = df.groupby('Primary_Modality')['J3_Score'].mean().rename('Mean_J3')
crosstab_body_j3 = df.groupby('Primary_Body_Region')['J3_Score'].mean().rename('Mean_J3')

df_p2=pd.DataFrame(list(stats_p2.items()),columns=['Statistic','Value'])
with pd.ExcelWriter(f'{OUT_SUPP}/P2_scanner_diversity.xlsx') as w:
    df_p2.to_excel(w, sheet_name='Metrics', index=False)
    crosstab_mod_j3.reset_index().to_excel(w, sheet_name='Modality_J3', index=False)
    crosstab_body_j3.reset_index().to_excel(w, sheet_name='BodyRegion_J3', index=False)
print('P2 done.'); df_p2


P2 done.


,Statistic,Value
0,Scanner-reporting rate,0.540863
1,Mean # manufacturers,2.345238
2,Median # manufacturers,1.500000
3,IQR # manufacturers,2.000000
4,Min # manufacturers,1.000000
5,Max # manufacturers,14.000000
6,Multi-vendor rate (J3>=3),0.268136
7,Multi-protocol testing rate,0.539945
8,Param disclosure rate,0.259871
9,Architecture transparency rate,0.570248


### 2.2 Kruskal-Wallis J3 across modalities + linear regression J3 ~ predictors

In [88]:
groups_mod=[g['J3_Score'].dropna().values for _,g in df.groupby('Primary_Modality')
             if g['J3_Score'].notna().sum()>=5]
if len(groups_mod)>=2:
    stat_kw,p_kw=scipy_stats.kruskal(*groups_mod)
    print(f'Kruskal-Wallis J3 x modality: H={stat_kw:.3f}, p={p_kw:.4f}')

df_lr=df.dropna(subset=['J3_Score','Published_Year','J2_Score'])
if len(df_lr)>20:
    lr_j3=smf.ols('J3_Score ~ Published_Year + C(Pathway) + C(Primary_Modality) + J2_Score',
                   data=df_lr).fit()
    print(lr_j3.summary())


Kruskal-Wallis J3 x modality: H=59.271, p=0.0000
                            OLS Regression Results                            
Dep. Variable:               J3_Score   R-squared:                       0.146
Model:                            OLS   Adj. R-squared:                  0.134
Method:                 Least Squares   F-statistic:                     11.49
Date:                Tue, 26 May 2026   Prob (F-statistic):           6.08e-28
Time:                        16:51:18   Log-Likelihood:                -2150.6
No. Observations:                1088   AIC:                             4335.
Df Residuals:                    1071   BIC:                             4420.
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------

### 2.3 Table 2 — Physics variance table

In [89]:
import re

phys_rows = []
for _, row in df.iterrows():
    f3 = str(row.get('F3-Acquisition_and_Protocols', '')).lower().strip()
    mod = row['Primary_Modality']
    
    kvp_detected = any(t in f3 for t in ['kv', 'kvp', 'kilovolt', 'voltage', 'energy'])
    slice_detected = any(t in f3 for t in ['slice', 'mm', 'thickness', 'thk', 'spacing', 'helical'])
    kernel_detected = any(t in f3 for t in ['kernel', 'filter', 'reconstruction', 'algorithm', 'sharp', 'smooth'])
    field_detected = any(t in f3 for t in ['1.5t', '3t', '3.0t', 'tesla', 'field strength', 'magnet', 'mhz', 'frequency'])

    has_params = (
        ("not reported" not in f3) and 
        ("parameters: nr" not in f3) and 
        (kvp_detected or slice_detected or kernel_detected or field_detected)
    )
    
    phys_rows.append({
        'Primary_Modality': mod,
        'kVp_reported': kvp_detected if mod in ['CT', 'X-ray', 'Mammography', 'DBT', 'Fluoroscopy'] else False, 
        'Slice_reported': slice_detected,
        'Kernel_reported': kernel_detected,
        'FieldStr_reported': field_detected if mod == 'MRI' else False, 
        'Multi_protocol': extract_multi_protocol(row.get('F3-Acquisition_and_Protocols', ''))
    })

df_phys=pd.DataFrame(phys_rows)
table2=df_phys.groupby('Primary_Modality')[['kVp_reported','Slice_reported','Kernel_reported',
    'FieldStr_reported','Multi_protocol']].mean().round(3)
table2.to_excel(f'{OUT_SUPP}/Table2_physics_variance.xlsx')
print('Table 2:'); print(table2)


Table 2:
                  kVp_reported  Slice_reported  Kernel_reported  \
Primary_Modality                                                  
CBCT                     0.000           0.000            0.000   
CT                       0.123           0.205            0.036   
DBT                      0.000           0.056            0.000   
Fluoroscopy              0.053           0.000            0.000   
MRI                      0.000           0.093            0.016   
Mammography              0.000           0.070            0.023   
Other                    0.000           0.000            0.000   
PET                      0.000           0.200            0.000   
PET/CT                   0.000           0.147            0.029   
SPECT                    0.000           0.111            0.000   
Ultrasound               0.000           0.016            0.010   
X-ray                    0.016           0.008            0.008   

                  FieldStr_reported  Multi_protocol 

### Fig 2A — Histogram: # manufacturers per device

In [90]:
fig2a,ax2a=plt.subplots(figsize=(7,4))
nb_valid=nb_mfr.dropna()
max_m=int(nb_valid.max()) if len(nb_valid)>0 else 5
ax2a.hist(nb_valid,bins=list(range(0,max_m+2)),align='left',color='steelblue',
          edgecolor='white',rwidth=0.8)
ax2a.set_xlabel('# scanner manufacturers tested')
ax2a.set_ylabel('# devices')
ax2a.set_title(f'Scanner manufacturers per device (N={len(nb_valid):,})')
plt.tight_layout()
fig2a.savefig(f'{OUT_SUPP}/Fig2A_mfr_histogram.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 2A saved.')


Fig 2A saved.


### Fig 2B — Box plot: J3 by modality

In [91]:
mod_ord=(df.groupby('Primary_Modality')['J3_Score'].median()
           .sort_values(ascending=False).index.tolist())
data_box=[df[df['Primary_Modality']==m]['J3_Score'].dropna().values for m in mod_ord]
fig2b,ax2b=plt.subplots(figsize=(max(8,len(mod_ord)*1.1),5))
bp=ax2b.boxplot(data_box,patch_artist=True,medianprops=dict(color='tomato',linewidth=2))
for patch in bp['boxes']: patch.set_facecolor('steelblue'); patch.set_alpha(0.6)
ax2b.set_xticklabels(mod_ord,rotation=40,ha='right',fontsize=9)
ax2b.set_ylabel('J3 Scanner Diversity Score (0-5)')
ax2b.set_title('J3 scanner diversity by imaging modality')
ax2b.axhline(3,color='tomato',linestyle='--',linewidth=1,label='Multi-vendor threshold')
ax2b.legend(fontsize=8)
plt.tight_layout()
fig2b.savefig(f'{OUT_SUPP}/Fig2B_J3_by_modality.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 2B saved.')


Fig 2B saved.


### Fig 2C — Manufacturer market share (treemap or bar)

In [92]:
if all_mfr:
    top_mfr=mfr_counts.head(10)
    try:
        import squarify
        fig2c,ax2c=plt.subplots(figsize=(10,6))
        colors_sq=plt.cm.tab10.colors[:len(top_mfr)]
        squarify.plot(sizes=top_mfr.values,
                      label=[f'{n}\n{v}' for n,v in top_mfr.items()],
                      color=colors_sq,alpha=0.8,ax=ax2c,text_kwargs={'fontsize':9})
        ax2c.axis('off'); ax2c.set_title('Scanner manufacturer share (treemap)')
    except ImportError:
        fig2c,ax2c=plt.subplots(figsize=(8,5))
        top_mfr.plot(kind='barh',ax=ax2c,color='steelblue')
        ax2c.set_xlabel('# validations'); ax2c.invert_yaxis()
        ax2c.set_title('Scanner manufacturer share (install squarify for treemap)')
    fig2c.savefig(f'{OUT_SUPP}/Fig2C_mfr_market_share.png',dpi=150,bbox_inches='tight')
    plt.show(); print('Fig 2C saved.')


Fig 2C saved.


### Fig 2D — Multi-vendor validation rate over time

In [93]:
mvr=(df[df['J3_Score'].notna()].groupby('Published_Year')['J3_Score']
       .apply(lambda x:(x>=3).mean()).reset_index()
       .rename(columns={'J3_Score':'MVR'}))
mvr['yr']=pd.to_numeric(mvr['Published_Year'],errors='coerce')
mvr=mvr.dropna(subset=['yr']).sort_values('yr')
fig2d,ax2d=plt.subplots(figsize=(9,4))
ax2d.plot(mvr['yr'],mvr['MVR'],marker='o',color='steelblue',linewidth=2)
ax2d.fill_between(mvr['yr'],mvr['MVR'],alpha=0.15,color='steelblue')
ax2d.set_ylim(0,1)
ax2d.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))
ax2d.set_xlabel('Published year'); ax2d.set_ylabel('% devices J3>=3')
ax2d.set_title('Multi-vendor validation rate over time')
plt.tight_layout()
fig2d.savefig(f'{OUT_SUPP}/Fig2D_multivend_over_time.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 2D saved.')


Fig 2D saved.


---
## Pillar 3 — The Validation Infrastructure Gap


### 3.1 Aggregation metrics

In [94]:
stats_p3={}

def _single_site(x):
    s=str(x).strip().lower()
    return s.startswith('1 ') or s=='1' or 'single' in s
def _multi_site(x):
    return any(opt in str(x) for opt in ['4-5','6-10','11-20','>20'])

for tok in ['510(K)','DE NOVO','PMA']:
    stats_p3[f'Pathway {tok}']=(df['Pathway'].str.upper()==tok).sum()/total_devices

stats_p3['Single-site rate']=df['D2-Number_of_Validation_Sites'].apply(_single_site).sum()/total_devices
stats_p3['Multi-site rate (>=4)']=df['D2-Number_of_Validation_Sites'].apply(_multi_site).sum()/total_devices

flat_vc=[c for lst in df['Val_Countries_Set'] for c in lst]
stats_p3['Validation country HHI']=(herfindahl_hirschman_index(
    pd.Series(flat_vc).value_counts().values) if flat_vc else float('nan'))

for design in ['Retrospective','Prospective','Both','Simulated','Literature-based']:
    stats_p3[f'Study design {design}']=(df['D1-Study_Design_and_Size'].apply(
        lambda x: design.lower() in str(x).lower()).sum()/total_devices)

ss=df['D1-Study_Design_and_Size'].apply(extract_sample_size).dropna()
stats_p3['Sample size Median']=ss.median()
stats_p3['Sample size IQR']=ss.quantile(0.75)-ss.quantile(0.25) if len(ss)>1 else float('nan')
stats_p3['Sample size Min']=ss.min(); stats_p3['Sample size Max']=ss.max()

stats_p3['Reader-study rate']=df['G2-Reader_Study_Design'].apply(
    lambda x: pd.notnull(x) and str(x).strip().upper() not in {'N/A','NR','NOT REPORTED'}
).sum()/total_devices

for comp in ['radiologist','standalone','predicate','ground truth','no comparator']:
    stats_p3[f'Comparator {comp}']=(
        df['G1-Comparator_Method'].apply(lambda x: comp in str(x).lower()).sum()/total_devices)

pav=df['Predicate_Age'].dropna()
stats_p3['Predicate age Mean']=pav.mean()
stats_p3['Predicate age Median']=pav.median()
stats_p3['Predicate age Max']=pav.max()
stats_p3['Chain depth Mean']=df['Predicate_Chain_Depth'].mean()
stats_p3['Chain depth Max']=df['Predicate_Chain_Depth'].max()

def _pred_creep(row):
    if pd.isnull(row['Published_Year']) or row['Published_Year']<2024: return 0
    ancestors=find_ancestor_chain(predicate_map,row['A1-Submission_Number'])
    if not ancestors: return 0
    root=ancestors[-1]
    rr=df[df['A1-Submission_Number'].astype(str).str.strip()==str(root).strip()]
    if rr.empty: return 0
    ry=pd.to_numeric(rr.iloc[0]['Published_Year'],errors='coerce')
    return 1 if pd.notnull(ry) and ry<2018 else 0

recs=df['Published_Year']>=2024
nr=recs.sum()
stats_p3['Predicate-creep score']=(df[recs].apply(_pred_creep,axis=1).sum()/nr if nr>0 else float('nan'))

df_p3=pd.DataFrame(list(stats_p3.items()),columns=['Statistic','Value'])
df_p3.to_excel(f'{OUT_SUPP}/P3_validation_infrastructure.xlsx',index=False)
print('P3 done.'); df_p3


P3 done.


,Statistic,Value
0,Pathway 510(K),9.834711e-01
1,Pathway DE NOVO,6.427916e-03
2,Pathway PMA,7.346189e-03
3,Single-site rate,2.203857e-02
4,Multi-site rate (>=4),1.248852e-01
5,Validation country HHI,3.739239e+03
6,Study design Retrospective,4.022039e-01
7,Study design Prospective,1.267218e-01
8,Study design Both,2.662994e-02
9,Study design Simulated,1.294766e-01


### 3.2 n=100 most recent devices — predicate chain subset

In [95]:
df_top100=(df.sort_values('Published_Year',ascending=False)
             .dropna(subset=['Published_Year']).head(100))
pa100=df_top100['Predicate_Age'].dropna()
pd100=df_top100['Predicate_Chain_Depth']
print(f'n=100 most recent: age mean={pa100.mean():.1f} median={pa100.median():.1f} max={pa100.max():.0f}')
print(f'chain depth mean={pd100.mean():.1f} max={pd100.max():.0f}')
df_top100[['A1-Submission_Number','Published_Year','Predicate_Age','Predicate_Chain_Depth']].to_excel(
    f'{OUT_SUPP}/P3_top100_predicate.xlsx',index=False)


n=100 most recent: age mean=2.9 median=1.5 max=13
chain depth mean=2.0 max=6


### 3.3 Correlation: predicate chain depth vs J2 score

In [96]:
df_corr=df[['Predicate_Chain_Depth','J2_Score']].dropna()
if len(df_corr)>5:
    r,pv=scipy_stats.pearsonr(df_corr['Predicate_Chain_Depth'],df_corr['J2_Score'])
    rs,ps=scipy_stats.spearmanr(df_corr['Predicate_Chain_Depth'],df_corr['J2_Score'])
    print(f'Pearson r={r:.3f} p={pv:.4f} | Spearman rho={rs:.3f} p={ps:.4f}')


Pearson r=0.113 p=0.0002 | Spearman rho=0.124 p=0.0000


### 3.4 Logistic regression — multi-site validation

In [97]:
df['Multi_Site_Flag']=df['D2-Number_of_Validation_Sites'].apply(_multi_site).astype(int)
logit_p3=fit_logistic_regression(df,'Multi_Site_Flag',
    ['Published_Year','Pathway','Primary_Modality','Primary_Body_Region'])
print(logit_p3.summary())


                           Logit Regression Results                           
Dep. Variable:        Multi_Site_Flag   No. Observations:                 1088
Model:                          Logit   Df Residuals:                     1060
Method:                           MLE   Df Model:                           27
Date:                Tue, 26 May 2026   Pseudo R-squ.:                  0.1198
Time:                        16:51:19   Log-Likelihood:                -360.83
converged:                      False   LL-Null:                       -409.93
Covariance Type:            nonrobust   LLR p-value:                 5.108e-10
                                               coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------
Intercept                                 -437.8468   5.97e+05     -0.001      0.999   -1.17e+06    1.17e+06
C(Pathway)[T.De Novo]                    

### 3.5 Kaplan-Meier-style survival curve — predicate ages

In [98]:
pred_ages_s=np.sort(pav.values); n_pa=len(pred_ages_s)
thr=range(0,int(pred_ages_s.max())+1)
km_y=[(pred_ages_s>=t).sum()/n_pa for t in thr]
fig_km,ax_km=plt.subplots(figsize=(8,4))
ax_km.step(list(thr),km_y,where='post',color='steelblue',linewidth=2)
ax_km.fill_between(list(thr),km_y,alpha=0.15,color='steelblue',step='post')
ax_km.set_xlabel('Predicate chain age (years)')
ax_km.set_ylabel('% devices with chain >= X years')
ax_km.set_title(f'Predicate age survival curve (N={n_pa:,})')
ax_km.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_:f'{y:.0%}'))
ax_km.axvline(7,color='tomato',linestyle='--',linewidth=1,label='7-year mark')
ax_km.legend(fontsize=8)
plt.tight_layout()
fig_km.savefig(f'{OUT_SUPP}/P3_predicate_KM_curve.png',dpi=150,bbox_inches='tight')
plt.show(); print('KM curve saved.')


KM curve saved.


### Fig 3A — D2 site count distribution

In [99]:
D2_ORDER=['1 (single-site)','2-3','4-5','6-10','11-20','>20','Not reported']
d2_n={}
for cat in D2_ORDER:
    if cat=='1 (single-site)':
        d2_n[cat]=df['D2-Number_of_Validation_Sites'].apply(_single_site).sum()
    elif cat=='Not reported':
        d2_n[cat]=df['D2-Number_of_Validation_Sites'].apply(
            lambda x: str(x).upper() in {'NR','NOT REPORTED','NAN'}).sum()
    else:
        d2_n[cat]=df['D2-Number_of_Validation_Sites'].apply(lambda x: cat in str(x)).sum()
fig3a,ax3a=plt.subplots(figsize=(9,4))
bars3a=ax3a.bar(D2_ORDER,[d2_n[c] for c in D2_ORDER],color='steelblue',edgecolor='white')
for bar,cat in zip(bars3a,D2_ORDER):
    ax3a.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.5,
              str(d2_n[cat]),ha='center',fontsize=9)
ax3a.set_xlabel('Number of validation sites'); ax3a.set_ylabel('Number of devices')
ax3a.set_title('D2: Validation site count distribution')
ax3a.tick_params(axis='x',rotation=20)
plt.tight_layout()
fig3a.savefig(f'{OUT_SUPP}/Fig3A_D2_site_distribution.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 3A saved.')


Fig 3A saved.


### Fig 3B — Predicate age histogram + chain depth inset

In [100]:
fig3b, ax3b = plt.subplots(figsize=(9, 5))
ax3b.hist(pav, bins=range(0, int(pav.max()) + 2), color='steelblue', edgecolor='white', linewidth=0.5)
ax3b.axvline(pav.median(), color='tomato', linestyle='--', linewidth=1.8,
             label=f'Median = {pav.median():.1f} yrs')
ax3b.set_xlabel('Predicate age (years)', fontsize=10)
ax3b.set_ylabel('# devices', fontsize=10)
ax3b.set_title(f'Predicate age distribution (N={len(pav):,})', fontsize=12, pad=15)
ax3b.legend(loc='upper left') 

ax_in = fig3b.add_axes([0.60, 0.45, 0.28, 0.38]) 

dc = df['Predicate_Chain_Depth'].value_counts().sort_index()
ax_in.bar(dc.index, dc.values, color='#f19333', edgecolor='white', width=0.7)
ax_in.set_xlabel('Chain depth', fontsize=8)
ax_in.set_ylabel('N', fontsize=8)
ax_in.set_title('Chain depth', fontsize=9, fontweight='bold')
ax_in.tick_params(labelsize=8)
fig3b.savefig(f'{OUT_MAIN}/Fig3B_predicate_age_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Fig 3B saved.')

Fig 3B saved.


### Fig 3C — Predicate chain network: representative chains

In [101]:
# Fig 3C — Representative predicate chain network visualization
try:
    import networkx as nx
    import matplotlib.patches as mpatches

    sample_sids = (df.sort_values('Published_Year', ascending=False)
                     .dropna(subset=['Published_Year'])
                     .head(200)['A1-Submission_Number'].tolist())

    chains_to_plot = []
    for sid in sample_sids:
        chain = find_ancestor_chain(predicate_map, sid)
        if len(chain) >= 2 and len(chains_to_plot) < 5:
            chains_to_plot.append([str(sid)] + [str(a) for a in chain])

    if not chains_to_plot:
        print('No chains with depth>=2 found for Fig 3C; skipping.')
    else:
        G = nx.DiGraph()
        chain_colors = ['#2980b9','#e67e22','#27ae60','#8e44ad','#c0392b']
        node_color_map = {}
        for ci, chain in enumerate(chains_to_plot):
            col = chain_colors[ci % len(chain_colors)]
            for i in range(len(chain) - 1):
                G.add_edge(chain[i+1], chain[i])   # ancestor -> descendant
            for node in chain:
                node_color_map[node] = col

        fig3c, ax3c = plt.subplots(figsize=(12, 6))
        pos = nx.spring_layout(G, seed=42, k=2.0)
        node_list = list(G.nodes())
        node_colors = [node_color_map.get(n, '#95a5a6') for n in node_list]
        nx.draw_networkx(G, pos, ax=ax3c, nodelist=node_list,
                         node_color=node_colors, node_size=600,
                         font_size=6, arrows=True,
                         arrowstyle='->', arrowsize=15,
                         edge_color='#7f8c8d', linewidths=0.5,
                         bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6))
        patches = [mpatches.Patch(color=chain_colors[i], label=f'Chain {i+1}')
                   for i in range(len(chains_to_plot))]
        ax3c.legend(handles=patches, fontsize=8, loc='upper left')
        ax3c.set_title(
            f'Representative predicate chains (n={len(chains_to_plot)} shown)',
            fontsize=11, fontweight='bold')
        ax3c.axis('off')
        plt.tight_layout()
        fig3c.savefig(f'{OUT_SUPP}/Fig3C_predicate_chain_network.png',
                      dpi=150, bbox_inches='tight')
        plt.show()
        print('Fig 3C saved.')
except ImportError:
    print('networkx not installed: pip install networkx')

Fig 3C saved.


### Fig 3E - Global choropleth of where validation actually happened

In [102]:
# Fig 3E — Global choropleth of where validation actually happened

COUNTRY_ISO3 = {
    'USA':'USA','UNITED STATES':'USA','US':'USA','CHINA':'CHN',
    'GERMANY':'DEU','FRANCE':'FRA','UNITED KINGDOM':'GBR','UK':'GBR',
    'SOUTH KOREA':'KOR','KOREA':'KOR','JAPAN':'JPN','INDIA':'IND',
    'CANADA':'CAN','AUSTRALIA':'AUS','NETHERLANDS':'NLD','ISRAEL':'ISR',
    'SWITZERLAND':'CHE','SWEDEN':'SWE','DENMARK':'DNK','BRAZIL':'BRA',
    'TURKEY':'TUR','MEXICO':'MEX','SOUTH AFRICA':'ZAF','ARGENTINA':'ARG',
    'INDONESIA':'IDN','ITALY':'ITA','SPAIN':'ESP','BELGIUM':'BEL',
    'AUSTRIA':'AUT','NORWAY':'NOR','FINLAND':'FIN','SINGAPORE':'SGP',
    'TAIWAN':'TWN','HONG KONG':'HKG',
}

flat_val = [c for lst in df['Val_Countries_Set'] for c in lst]
val_cc = pd.Series(flat_val).value_counts().reset_index()
val_cc.columns = ['Country_Raw', 'Count']
val_cc['ISO3'] = val_cc['Country_Raw'].str.upper().map(COUNTRY_ISO3)
val_cc = val_cc.dropna(subset=['ISO3'])
val_cc.to_excel(f'{OUT_SUPP}/P3_Fig3E_choropleth_data.xlsx', index=False)

try:
    import plotly.express as px
    fig3e = px.choropleth(
        val_cc, locations='ISO3', color='Count',
        hover_name='Country_Raw',
        color_continuous_scale='Blues',
        title='Global distribution of AI validation sites (D3)',
        labels={'Count': '# device validations'}
    )
    fig3e.update_layout(
        geo=dict(showframe=False, showcoastlines=True),
        coloraxis_colorbar=dict(title='# validations')
    )
    fig3e.write_image(f'{OUT_MAIN}/Fig3E_validation_choropleth.png',
                      width=1200, height=600)
    print('Fig 3E choropleth saved to OUT_MAIN.')
except ImportError:
    vc_bar = val_cc.nlargest(20, 'Count').set_index('Country_Raw')['Count']
    fig3e, ax3e = plt.subplots(figsize=(10, 5))
    vc_bar.plot(kind='barh', ax=ax3e, color='steelblue')
    ax3e.invert_yaxis()
    ax3e.set_xlabel('# device validations')
    ax3e.set_title('Top 20 validation countries — install plotly+kaleido for choropleth')
    plt.tight_layout()
    fig3e.savefig(f'{OUT_MAIN}/Fig3E_validation_choropleth.png',
                  dpi=150, bbox_inches='tight')
    plt.show()
    print('Fig 3E fallback bar chart saved to OUT_MAIN.')

Fig 3E choropleth saved to OUT_MAIN.


### Fig 3D — Sunburst: Pathway -> Study design -> Comparator

In [135]:
df_sun=df[['Pathway','D1-Study_Design_and_Size','G1-Comparator_Method']].copy()
df_sun['Study_Design']=df_sun['D1-Study_Design_and_Size'].apply(
    lambda x: next((d for d in ['Retrospective','Prospective','Both','Simulated']
                    if d.lower() in str(x).lower()),'Not described'))
df_sun['Comparator'] = df['G1-Comparator_Method'].apply(clean_comparator_method)
df_sun_grp=(df_sun.groupby(['Pathway','Study_Design','Comparator'])
                   .size().reset_index(name='Count'))
df_sun_grp.to_excel(f'{OUT_SUPP}/P3_Fig3D_sunburst_data.xlsx',index=False)
try:
    import plotly.express as px

    fig3d = px.sunburst(
        df_sun_grp,
        path=['Pathway', 'Study_Design', 'Comparator'],
        values='Count',
        title='Pathway → Study Design → Comparator Method',
        color='Study_Design',
        color_discrete_sequence=px.colors.qualitative.Pastel
    )
    
    fig3d.update_traces(
        textinfo="label+percent parent",
        insidetextorientation="radial"
    )
    
    fig3d.update_layout(
        margin=dict(t=40, l=20, r=20, b=20),
        width=800,
        height=800
    )
    
    fig3d.write_image(f'{OUT_SUPP}/Fig3D_sunburst.png', scale=2)
    print('Fig 3D saved (plotly optimized).')
except ImportError:
    print('Fig 3D: pip install plotly kaleido.')


Fig 3D saved (plotly optimized).


### Table 3 — Cross-jurisdictional regulatory comparison

In [104]:
table3=pd.DataFrame([
    {'Jurisdiction':'FDA (USA)','Training disclosure':'No (encouraged)',
     'Validation demographics':'No','Multi-site required':'No',
     'Predicate reliance':'510(k) allowed','PCCP framework':'Yes (2023)'},
    {'Jurisdiction':'Health Canada','Training disclosure':'Partial (2026)',
     'Validation demographics':'Partial','Multi-site required':'Recommended',
     'Predicate reliance':'Limited','PCCP framework':'Proposed'},
    {'Jurisdiction':'EU AI Act','Training disclosure':'Yes (high-risk)',
     'Validation demographics':'Yes','Multi-site required':'Recommended',
     'Predicate reliance':'N/A','PCCP framework':'Lifecycle monitoring'},
    {'Jurisdiction':'TGA (Australia)','Training disclosure':'No',
     'Validation demographics':'No','Multi-site required':'No',
     'Predicate reliance':'Similar to FDA','PCCP framework':'Under review'},
    {'Jurisdiction':'PMDA (Japan)','Training disclosure':'Partial',
     'Validation demographics':'No','Multi-site required':'No',
     'Predicate reliance':'Limited','PCCP framework':'Pilot (2024)'},
])
table3.to_excel(f'{OUT_SUPP}/Table3_cross_jurisdictional.xlsx',index=False)
print('Table 3 saved.'); table3


Table 3 saved.


,Jurisdiction,Training disclosure,Validation demographics,Multi-site required,Predicate reliance,PCCP framework
0,FDA (USA),No (encouraged),No,No,510(k) allowed,Yes (2023)
1,Health Canada,Partial (2026),Partial,Recommended,Limited,Proposed
2,EU AI Act,Yes (high-risk),Yes,Recommended,N/A,Lifecycle monitoring
3,TGA (Australia),No,No,No,Similar to FDA,Under review
4,PMDA (Japan),Partial,No,No,Limited,Pilot (2024)


---
## Pillar 4 — Cross-Border Data Challenges


In [105]:
stats_p4={}
hq4=df[df['HQ_Country']!='NR']['HQ_Country'].value_counts()
for c,n in hq4.items(): stats_p4[f'HQ % {c}']=n/total_devices

def _align(row):
    t,v=set(row['Train_Countries_Set']),set(row['Val_Countries_Set'])
    return 1 if t and v and t.issubset(v) else 0

stats_p4['Training-val alignment']=df.apply(_align,axis=1).sum()/total_devices
stats_p4['Cross-border training']=df['Train_Countries_Set'].apply(lambda x:len(set(x))>1).sum()/total_devices
stats_p4['Cross-border validation']=df['Val_Countries_Set'].apply(lambda x:len(set(x))>1).sum()/total_devices
stats_p4['HQ!=val country']=df.apply(
    lambda r:1 if r['HQ_Country']!='NR' and set(r['Val_Countries_Set'])
    and r['HQ_Country'] not in set(r['Val_Countries_Set']) else 0,axis=1).sum()/total_devices
stats_p4['Public dataset use']=df['C1b-Training_Data_Source'].apply(is_public_dataset).sum()/total_devices

pair_c=Counter()
for _,row in df.iterrows():
    for tc in row['Train_Countries_Set']:
        for vc in row['Val_Countries_Set']: pair_c[(tc,vc)]+=1
df_pairs=pd.DataFrame([{'Train_Country':t,'Val_Country':v,'Count':c}
                         for (t,v),c in pair_c.most_common()])

if len(df_pairs)>0:
    top1_pct=df_pairs.iloc[0]['Count']/total_devices
    print(f"Top pair: {df_pairs.iloc[0]['Train_Country']}->{df_pairs.iloc[0]['Val_Country']} = {top1_pct:.1%}")
    stats_p4['Top pair % of devices']=top1_pct

df_p4=pd.DataFrame(list(stats_p4.items()),columns=['Statistic','Value'])
with pd.ExcelWriter(f'{OUT_SUPP}/P4_cross_border.xlsx') as w:
    df_p4.to_excel(w,sheet_name='Metrics',index=False)
    df_pairs.to_excel(w,sheet_name='Bilateral_pairs',index=False)
    if len(df_pairs)>0: df_pairs.head(10).to_excel(w,sheet_name='Top10_pairs',index=False)
print('P4 done.'); df_p4


Top pair: UNITED STATES->UNITED STATES = 22.3%
P4 done.


,Statistic,Value
0,HQ % UNITED STATES,0.152433
1,HQ % GERMANY,0.055096
2,HQ % FRANCE,0.031221
3,HQ % REPUBLIC OF KOREA,0.022957
4,HQ % NETHERLANDS,0.019284
5,HQ % CHINA,0.018365
6,HQ % JAPAN,0.014692
7,HQ % ISRAEL,0.014692
8,HQ % UNITED KINGDOM,0.012856
9,HQ % SWEDEN,0.007346


### Fig 4A — Cross-border data flow Sankey

In [106]:
try:
    import plotly.graph_objects as go

    TOP_N = 8  # cap nodes per layer for readability
    palette = ['rgba(41,128,185,0.55)','rgba(39,174,96,0.55)',
               'rgba(231,76,60,0.55)', 'rgba(142,68,173,0.55)',
               'rgba(230,126,34,0.55)','rgba(52,152,219,0.55)',
               'rgba(26,188,156,0.55)','rgba(192,57,43,0.55)']

    top_hq = df['HQ_Country'].value_counts().head(TOP_N).index.tolist()
    top_tc = (pd.Series([c for lst in df['Train_Countries_Set'] for c in lst])
               .value_counts().head(TOP_N).index.tolist())
    top_vc = (pd.Series([c for lst in df['Val_Countries_Set'] for c in lst])
               .value_counts().head(TOP_N).index.tolist())

    all_nodes = ([f'HQ: {c}'    for c in top_hq]
               + [f'Train: {c}' for c in top_tc]
               + [f'Val: {c}'   for c in top_vc])
    nidx = {n: i for i, n in enumerate(all_nodes)}

    src_list, tgt_list, val_list, color_list = [], [], [], []

    # HQ -> training country
    for _, row in df.iterrows():
        hq = row['HQ_Country']
        if hq not in top_hq: continue
        for tc in row['Train_Countries_Set']:
            if tc not in top_tc: continue
            src_list.append(nidx[f'HQ: {hq}'])
            tgt_list.append(nidx[f'Train: {tc}'])
            val_list.append(1)
            color_list.append(palette[top_hq.index(hq) % len(palette)])

    # Training country -> validation country
    for _, row in df.iterrows():
        for tc in row['Train_Countries_Set']:
            if tc not in top_tc: continue
            for vc in row['Val_Countries_Set']:
                if vc not in top_vc: continue
                src_list.append(nidx[f'Train: {tc}'])
                tgt_list.append(nidx[f'Val: {vc}'])
                val_list.append(1)
                color_list.append('rgba(127,140,141,0.35)')

    node_colors = (
        [palette[i % len(palette)] for i in range(len(top_hq))]
        + ['rgba(39,174,96,0.8)']  * len(top_tc)
        + ['rgba(192,57,43,0.8)']  * len(top_vc)
    )

    fig4a = go.Figure(go.Sankey(
        arrangement='snap',
        node=dict(label=all_nodes, pad=18, thickness=20, color=node_colors),
        link=dict(source=src_list, target=tgt_list, value=val_list, color=color_list)
    ))
    fig4a.update_layout(
        title_text='Cross-border data flow: Company HQ → Training → Validation',
        font_size=11, height=550
    )
    fig4a.write_image(f'{OUT_MAIN}/Fig4A_sankey_HQ_train_val.png',
                      width=1200, height=600)
    print('Fig 4A Sankey saved to OUT_MAIN.')
except ImportError:
    print('Fig 4A requires: pip install plotly kaleido')

Fig 4A Sankey saved to OUT_MAIN.


### Fig 4B — Heatmap: training x validation country matrix

In [107]:
if len(df_pairs)>0:
    top_t=df_pairs.groupby('Train_Country')['Count'].sum().nlargest(12).index
    top_v=df_pairs.groupby('Val_Country')['Count'].sum().nlargest(12).index
    df_hm4=(df_pairs[df_pairs['Train_Country'].isin(top_t)&df_pairs['Val_Country'].isin(top_v)]
             .pivot_table(index='Train_Country',columns='Val_Country',values='Count',fill_value=0))
    fig4b,ax4b=plt.subplots(figsize=(max(8,len(top_v)),max(6,len(top_t)*0.6)))
    im4b=ax4b.imshow(df_hm4.values,cmap='Blues',aspect='auto')
    ax4b.set_xticks(range(len(df_hm4.columns)))
    ax4b.set_xticklabels(df_hm4.columns,rotation=40,ha='right',fontsize=8)
    ax4b.set_yticks(range(len(df_hm4.index))); ax4b.set_yticklabels(df_hm4.index,fontsize=8)
    for i in range(len(df_hm4.index)):
        for j in range(len(df_hm4.columns)):
            val=df_hm4.values[i,j]
            if val>0:
                ax4b.text(j,i,str(int(val)),ha='center',va='center',fontsize=7,
                          color='white' if val>df_hm4.values.max()*0.5 else 'black')
    plt.colorbar(im4b,ax=ax4b,label='# devices')
    ax4b.set_xlabel('Validation country'); ax4b.set_ylabel('Training country')
    ax4b.set_title('Bilateral training x validation country matrix (top 12)')
    plt.tight_layout()
    fig4b.savefig(f'{OUT_SUPP}/Fig4B_country_pair_heatmap.png',dpi=150,bbox_inches='tight')
    plt.show(); print('Fig 4B saved.')


Fig 4B saved.


### Fig 4C — World map with arcs: HQ to validation country

In [136]:
COUNTRY_COORDS = {
    'USA':{'lat':38,'lon':-97},'UNITED STATES':{'lat':38,'lon':-97},
    'CHINA':{'lat':35,'lon':105},'GERMANY':{'lat':51,'lon':10},
    'FRANCE':{'lat':46,'lon':2},'UNITED KINGDOM':{'lat':54,'lon':-2},
    'UK':{'lat':54,'lon':-2},'SOUTH KOREA':{'lat':37,'lon':128},
    'KOREA':{'lat':37,'lon':128},'JAPAN':{'lat':36,'lon':138},
    'INDIA':{'lat':20,'lon':78},'CANADA':{'lat':56,'lon':-96},
    'AUSTRALIA':{'lat':-25,'lon':133},'NETHERLANDS':{'lat':52,'lon':5},
    'ISRAEL':{'lat':31,'lon':35},'SWITZERLAND':{'lat':47,'lon':8},
    'SWEDEN':{'lat':62,'lon':16},'DENMARK':{'lat':56,'lon':10},
    'BRAZIL':{'lat':-15,'lon':-47},'ITALY':{'lat':42,'lon':12},
    'SPAIN':{'lat':40,'lon':-4},'BELGIUM':{'lat':50,'lon':4},
    'AUSTRIA':{'lat':47,'lon':14},'SINGAPORE':{'lat':1,'lon':104},
    'TAIWAN':{'lat':24,'lon':121},'NORWAY':{'lat':60,'lon':8},
    'REPUBLIC OF KOREA': {'lat':37, 'lon':128},
}

try:
    import plotly.graph_objects as go

    flow_pairs = {}
    for _, row in df.iterrows():
        hq = str(row['HQ_Country']).upper()
        if hq not in COUNTRY_COORDS: continue
        for vc in row['Val_Countries_Set']:
            if vc not in COUNTRY_COORDS or vc == hq: continue
            flow_pairs[(hq, vc)] = flow_pairs.get((hq, vc), 0) + 1

    top_flows = sorted(flow_pairs.items(), key=lambda x: -x[1])[:30]
    max_flow = max(v for _, v in top_flows) if top_flows else 1

    traces = []
    for (hq, vc), cnt in top_flows:
        h, v_c = COUNTRY_COORDS[hq], COUNTRY_COORDS[vc]
        traces.append(go.Scattergeo(
            lon=[h['lon'], v_c['lon']], lat=[h['lat'], v_c['lat']],
            mode='lines',
            line=dict(width=0.5 + 3.5 * (cnt / max_flow),
                      color='rgba(41,128,185,0.5)'),
            hoverinfo='none', showlegend=False
        ))

    hq_counts = df['HQ_Country'].value_counts()
    valid_hq = [c for c in hq_counts.index if c in COUNTRY_COORDS]
    traces.append(go.Scattergeo(
        lon=[COUNTRY_COORDS[c]['lon'] for c in valid_hq],
        lat=[COUNTRY_COORDS[c]['lat'] for c in valid_hq],
        text=[f'{c}: {hq_counts[c]} devices' for c in valid_hq],
        mode='markers',
        marker=dict(
            size=[3 + 12 * hq_counts[c] / hq_counts.max() for c in valid_hq],
            color='rgba(192,57,43,0.8)',
            line=dict(width=0.5, color='white')
        ),
        hovertemplate='%{text}', name='HQ'
    ))

    fig4c = go.Figure(traces)
    fig4c.update_layout(
        title='Device HQ → validation country flows (top 30 cross-border pairs)',
        geo=dict(showland=True, landcolor='rgb(240,240,240)',
                 showocean=True, oceancolor='rgb(220,235,248)',
                 showcoastlines=True, coastlinecolor='rgb(180,180,180)',
                 projection_type='natural earth'),
        height=520
    )
    fig4c.write_image(f'{OUT_SUPP}/Fig4C_arc_map_HQ_to_val.png',
                      width=1200, height=550)
    print('Fig 4C arc map saved to OUT_SUPP.')
except ImportError:
    print('Fig 4C requires: pip install plotly kaleido')

Fig 4C arc map saved to OUT_SUPP.


### Table 4 — Counterargument table

In [109]:
table4=pd.DataFrame([
    {'Counterargument':'Small samples prevent subgroup analysis',
     'Empirical response':'Median validation N is large; absence reflects design choice not power'},
    {'Counterargument':'Race/ethnicity categories are socially constructed',
     'Empirical response':'Imprecision does not justify omission; proxies improve equity monitoring'},
    {'Counterargument':'Demographic data not captured at point of imaging',
     'Empirical response':'EHR linkage is routine in multi-site studies; barrier is institutional'},
    {'Counterargument':'Mandatory reporting increases regulatory burden',
     'Empirical response':'De Novo/PMA already report richer data; standardisation reduces burden'},
    {'Counterargument':'Performance gaps not clinically meaningful',
     'Empirical response':'Literature shows >10pp AUC drops across demographic strata'},
])
table4.to_excel(f'{OUT_SUPP}/Table4_counterarguments.xlsx',index=False)
print('Table 4 saved.'); table4


Table 4 saved.


,Counterargument,Empirical response
0,Small samples prevent subgroup analysis,Median validation N is large; absence reflects...
1,Race/ethnicity categories are socially constru...,Imprecision does not justify omission; proxies...
2,Demographic data not captured at point of imaging,EHR linkage is routine in multi-site studies; ...
3,Mandatory reporting increases regulatory burden,De Novo/PMA already report richer data; standa...
4,Performance gaps not clinically meaningful,Literature shows >10pp AUC drops across demogr...


---
## Pillar 5 — Workforce Equity and Talent Pipelines


In [110]:
stats_p5={}
hq_nonr=df[df['HQ_Country']!='NR']['HQ_Country'].value_counts()
stats_p5['HQ Gini']=calculate_gini(hq_nonr.values) if len(hq_nonr)>0 else float('nan')
stats_p5['% high-income']=(df['Income_Bracket']=='High-Income').sum()/total_devices
stats_p5['% LMICs']=(df['Income_Bracket']=='LMIC').sum()/total_devices
for st in ['academic','community','mixed']:
    stats_p5[f'Site {st}']=df['Val_Types_Clean'].apply(lambda x:st in x).sum()/total_devices
stats_p5['% academic-only']=df['Val_Types_Clean'].apply(
    lambda x:'academic' in x and 'community' not in x and 'mixed' not in x).sum()/total_devices
crosstab_k1=pd.crosstab(df['K1-Document_Quality'],df['HQ_Region'],normalize='columns')
df_p5=pd.DataFrame(list(stats_p5.items()),columns=['Statistic','Value'])
with pd.ExcelWriter(f'{OUT_SUPP}/P5_workforce_equity.xlsx') as w:
    df_p5.to_excel(w,sheet_name='Metrics',index=False)
    crosstab_k1.to_excel(w,sheet_name='K1_x_HQ_Region')
print('P5 done.'); df_p5


P5 done.


,Statistic,Value
0,HQ Gini,0.660163
1,% high-income,0.320478
2,% LMICs,0.020202
3,Site academic,0.025712
4,Site community,0.013774
5,Site mixed,0.049587
6,% academic-only,0.025712


### Fig 5A — Global choropleth of company HQ concentration

In [111]:
COUNTRY_ISO3_P5 = {
    'USA':'USA','UNITED STATES':'USA','CHINA':'CHN','GERMANY':'DEU',
    'FRANCE':'FRA','UNITED KINGDOM':'GBR','UK':'GBR',
    'SOUTH KOREA':'KOR','KOREA':'KOR','JAPAN':'JPN','INDIA':'IND',
    'CANADA':'CAN','AUSTRALIA':'AUS','NETHERLANDS':'NLD','ISRAEL':'ISR',
    'SWITZERLAND':'CHE','SWEDEN':'SWE','DENMARK':'DNK','BRAZIL':'BRA',
    'ITALY':'ITA','SPAIN':'ESP','BELGIUM':'BEL','AUSTRIA':'AUT',
    'NORWAY':'NOR','FINLAND':'FIN','SINGAPORE':'SGP','TAIWAN':'TWN',
    'TURKEY':'TUR','MEXICO':'MEX','SOUTH AFRICA':'ZAF','ARGENTINA':'ARG',
    'INDONESIA':'IDN',
}

hq_5a = (df[df['HQ_Country'] != 'NR']['HQ_Country']
          .value_counts().reset_index())
hq_5a.columns = ['Country_Raw', 'N_Devices']
hq_5a['ISO3'] = hq_5a['Country_Raw'].str.upper().map(COUNTRY_ISO3_P5)
hq_5a = hq_5a.dropna(subset=['ISO3'])
hq_5a.to_excel(f'{OUT_SUPP}/P5_Fig5A_choropleth_data.xlsx', index=False)

try:
    import plotly.express as px
    fig5a = px.choropleth(
        hq_5a, locations='ISO3', color='N_Devices',
        hover_name='Country_Raw',
        color_continuous_scale='YlOrRd',
        title='Company HQ concentration by country (B4)',
        labels={'N_Devices': '# cleared devices'}
    )
    fig5a.update_layout(
        geo=dict(showframe=False, showcoastlines=True),
        coloraxis_colorbar=dict(title='# devices')
    )
    fig5a.write_image(f'{OUT_SUPP}/Fig5A_HQ_choropleth.png',
                      width=1200, height=600)
    print('Fig 5A saved to OUT_SUPP.')
except ImportError:
    hq_bar = hq_5a.nlargest(20, 'N_Devices').set_index('Country_Raw')['N_Devices']
    fig5a, ax5a = plt.subplots(figsize=(10, 5))
    hq_bar.plot(kind='barh', ax=ax5a, color='#e67e22')
    ax5a.invert_yaxis()
    ax5a.set_xlabel('# devices')
    ax5a.set_title('Company HQ counts (top 20) — install plotly+kaleido for choropleth')
    plt.tight_layout()
    fig5a.savefig(f'{OUT_SUPP}/Fig5A_HQ_choropleth.png',
                  dpi=150, bbox_inches='tight')
    plt.show()
    print('Fig 5A fallback bar chart saved to OUT_SUPP.')

Fig 5A saved to OUT_SUPP.


### Fig 5B — Lorenz curve + Gini

In [112]:
x_lor,y_lor=lorenz_curve(hq_nonr.values)
gini_val=stats_p5['HQ Gini']
fig5b=plot_lorenz_curve(x_lor,y_lor,gini=gini_val,
    title='Lorenz curve - company HQ concentration',
    save_path=f'{OUT_SUPP}/Fig5B_lorenz.png')
plt.show(); print(f'Fig 5B saved. Gini={gini_val:.3f}')


Fig 5B saved. Gini=0.660


### 5.3 Ordinal regression — K1 ~ pathway + HQ region

In [113]:
ord_k1=fit_ordinal_regression(df,'K1-Document_Quality',
    ['Opaque','Sparse','Moderate','Rich'],
    ['Pathway','HQ_Region','Published_Year'])
print(ord_k1.summary())


                              OrderedModel Results                             
Dep. Variable:     K1-Document_Quality   Log-Likelihood:                -1381.0
Model:                    OrderedModel   AIC:                             2786.
Method:             Maximum Likelihood   BIC:                             2846.
Date:                 Tue, 26 May 2026                                         
Time:                         16:51:32                                         
No. Observations:                 1088                                         
Df Residuals:                     1076                                         
Df Model:                            9                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Pathway_De Novo             0.7798      0.653      1.194      0.232      -0.500       2.059
Path

### Fig 5C — Stacked bar: site types by year and by pathway

In [139]:
for grouper,fname in [('Published_Year','year'),('Pathway','pathway')]:
    df_filtered = df[~df[grouper].astype(str).str.upper().isin(['NR', 'NOT REPORTED', 'NAN'])]
    
    site_dist = (df_filtered.groupby(grouper)['Val_Types_Clean'].apply(
        lambda g: pd.Series({
            'Academic':       g.apply(lambda x: 'academic' in str(x).lower() and 'community' not in str(x).lower() and 'mixed' not in str(x).lower()).mean(),
            'Mixed':          g.apply(lambda x: 'mixed' in str(x).lower()).mean(),
            'Community':      g.apply(lambda x: 'community' in str(x).lower() and 'academic' not in str(x).lower()).mean(),
            'Not Disclosed':  g.apply(lambda x: 'nr' in str(x).lower() or pd.isna(x)).mean(),
        })).unstack())
    
    fig5c,ax5c=plt.subplots(figsize=(max(8,len(site_dist)*1.1),5))
    site_dist.plot(kind='bar', stacked=True, ax=ax5c,
               color=['#2980b9', '#e67e22', '#27ae60', '#7f8c8d'], edgecolor='white')
    ax5c.set_ylim(0,1); ax5c.set_ylabel('Proportion')
    ax5c.set_title(f'Validation site types by {fname}')
    ax5c.tick_params(axis='x',rotation=30); ax5c.legend(title='Site type',fontsize=8)
    plt.tight_layout()
    fig5c.savefig(f'{OUT_SUPP}/Fig5C_site_types_by_{fname}.png',dpi=150,bbox_inches='tight')
    plt.show()
print('Fig 5C saved.')


Fig 5C saved.


### Table 5 — Training-program inventory

In [115]:
table5=pd.DataFrame([
    {'Program':'CAMERA','Region':'Africa','Focus':'Clinical AI & radiology','Trainees/yr':'~200'},
    {'Program':'SPARK','Region':'Africa','Focus':'AI research capacity','Trainees/yr':'~150'},
    {'Program':'SWIM','Region':'Multi-regional (LMIC)','Focus':'Women in medical imaging','Trainees/yr':'~100'},
    {'Program':'RSNA AI Course','Region':'Global (online)','Focus':'Radiology AI literacy','Trainees/yr':'>5000'},
    {'Program':'ESR AI curriculum','Region':'Europe','Focus':'Residency AI integration','Trainees/yr':'varies'},
])
table5.to_excel(f'{OUT_SUPP}/Table5_training_programs.xlsx',index=False)
print('Table 5 saved.'); table5


Table 5 saved.


,Program,Region,Focus,Trainees/yr
0,CAMERA,Africa,Clinical AI & radiology,~200
1,SPARK,Africa,AI research capacity,~150
2,SWIM,Multi-regional (LMIC),Women in medical imaging,~100
3,RSNA AI Course,Global (online),Radiology AI literacy,>5000
4,ESR AI curriculum,Europe,Residency AI integration,varies


---
## Pillar 6 — Post-Market Surveillance


In [116]:
df['Has_PCCP']=df['H2-PCCP_Filed'].apply(
    lambda x:1 if any(w in str(x).upper() for w in ['AUTHORIZED','FILED','YES']) else 0)
df['Recall_Flag']=df['H3-Regulatory_Flags'].apply(lambda x:extract_h3_subflag(x,'Recall:'))
df['Expedited_Flag']=df['H3-Regulatory_Flags'].apply(lambda x:extract_h3_subflag(x,'Expedited:'))
df['Third_Party_Flag']=df['H3-Regulatory_Flags'].apply(lambda x:extract_h3_subflag(x,'Third party:'))
df['NCT_Flag']=df['H3-Regulatory_Flags'].apply(lambda x:extract_h3_subflag(x,'NCT:'))

stats_p6={}
stats_p6['PCCP adoption']=df['Has_PCCP'].sum()/total_devices
stats_p6['Recall rate']=df['Recall_Flag'].isin(['YES','CHECK DATABASE']).sum()/total_devices
stats_p6['Breakthrough rate']=df['Expedited_Flag'].str.contains('YES',na=False).sum()/total_devices
stats_p6['Third-party review']=df['Third_Party_Flag'].str.contains('YES',na=False).sum()/total_devices
stats_p6['NCT-linked']=df['NCT_Flag'].apply(
    lambda x:str(x).upper() not in {'N/A','NO','NR',''} and len(str(x))>2).sum()/total_devices

def _cascade(row):
    if pd.isnull(row['Published_Year']) or row['Published_Year']<2024: return 0
    for anc in set(find_ancestor_chain(predicate_map,row['A1-Submission_Number'])):
        ar=df[df['A1-Submission_Number'].astype(str).str.strip()==str(anc).strip()]
        if not ar.empty and ar.iloc[0]['Recall_Flag'].upper() in ['YES','CHECK DATABASE']:
            return 1
    return 0

nr6=(df['Published_Year']>=2024).sum()
stats_p6['Predicate-cascade risk']=(df[df['Published_Year']>=2024]
    .apply(_cascade,axis=1).sum()/nr6 if nr6>0 else float('nan'))

pccp_yearly=(df[df['Published_Year']>=2023].groupby('Published_Year')['Has_PCCP']
               .mean().reset_index().rename(columns={'Has_PCCP':'PCCP_rate'}))
crosstab_pccp=pd.crosstab(df['Pathway'],df['H2-PCCP_Filed'],normalize='index')

df_p6=pd.DataFrame(list(stats_p6.items()),columns=['Statistic','Value'])
with pd.ExcelWriter(f'{OUT_SUPP}/P6_post_market.xlsx') as w:
    df_p6.to_excel(w,sheet_name='Metrics',index=False)
    pccp_yearly.to_excel(w,sheet_name='PCCP_timeseries',index=False)
    crosstab_pccp.to_excel(w,sheet_name='PCCP_x_Pathway')
print('P6 done.'); df_p6


P6 done.


,Statistic,Value
0,PCCP adoption,0.037649
1,Recall rate,0.229568
2,Breakthrough rate,0.000918
3,Third-party review,0.376492
4,NCT-linked,0.001837
5,Predicate-cascade risk,0.210162


### 6.2 Logistic regression — PCCP ~ pathway + modality + year + J4

In [117]:
logit_p6=fit_logistic_regression(df,'Has_PCCP',
    ['Published_Year','Pathway','Primary_Modality','J4_Composite_Score'])
print(logit_p6.summary())


                           Logit Regression Results                           
Dep. Variable:               Has_PCCP   No. Observations:                 1088
Model:                          Logit   Df Residuals:                     1071
Method:                           MLE   Df Model:                           16
Date:                Tue, 26 May 2026   Pseudo R-squ.:                  0.2029
Time:                        16:51:32   Log-Likelihood:                -136.62
converged:                      False   LL-Null:                       -171.38
Covariance Type:            nonrobust   LLR p-value:                 1.204e-08
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Intercept                          -1337.9449   1.16e+05     -0.012      0.991   -2.28e+05    2.25e+05
C(Pathway)[T.De Novo]                  3.6806      0.920   

### 6.3 Table 6 — Recall cascade table

In [118]:
recalled=df[df['Recall_Flag'].isin(['YES','CHECK DATABASE'])]['A1-Submission_Number'].tolist()
cascade_rows=[]
for rsid in recalled:
    ds=df[df['A1-Submission_Number'].apply(
        lambda sid: rsid in find_ancestor_chain(predicate_map,sid))]['A1-Submission_Number'].tolist()
    cascade_rows.append({'Recalled ancestor':rsid,'N downstream':len(ds),
        'Downstream (first 5)':'; '.join(ds[:5])+('...' if len(ds)>5 else '')})
df_cascade=pd.DataFrame(cascade_rows)
df_cascade.to_excel(f'{OUT_SUPP}/Table6_recall_cascade.xlsx',index=False)
print(f'Table 6: {len(df_cascade)} recalled ancestors.'); df_cascade


Table 6: 250 recalled ancestors.


,Recalled ancestor,N downstream,Downstream (first 5)
0,K253091,0,
1,K252235,0,
2,K250766,0,
3,K252482,0,
4,K251151,0,
...,...,...,...
245,K233657,1,K243523
246,K242130,0,
247,K241719,0,
248,K243523,0,


### Fig 6A — PCCP adoption rate over time

In [119]:
pccp_plot=pccp_yearly.copy()
pccp_plot['yr']=pd.to_numeric(pccp_plot['Published_Year'],errors='coerce')
pccp_plot=pccp_plot.dropna(subset=['yr']).sort_values('yr')
fig6a,ax6a=plt.subplots(figsize=(9,4))
ax6a.plot(pccp_plot['yr'],pccp_plot['PCCP_rate'],marker='o',color='steelblue',linewidth=2)
ax6a.fill_between(pccp_plot['yr'],pccp_plot['PCCP_rate'],alpha=0.15,color='steelblue')
for yr,lbl in {2023:'FDA PCCP\nguidance',2024:'PCCP expansion'}.items():
    ax6a.axvline(yr,color='tomato',linestyle='--',linewidth=1,alpha=0.7)
    ax6a.text(yr+0.05,0.95,lbl,fontsize=7,color='tomato',va='top',transform=ax6a.get_xaxis_transform())
ax6a.set_ylim(0,1)
ax6a.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_:f'{y:.0%}'))
ax6a.set_xlabel('Published year'); ax6a.set_ylabel('PCCP rate')
ax6a.set_title('PCCP adoption rate over time (post-2023)')
plt.tight_layout()
fig6a.savefig(f'{OUT_SUPP}/Fig6A_PCCP_timeseries.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 6A saved.')


Fig 6A saved.


### Fig 6B — H3 regulatory flag frequencies

In [120]:
h3_flags={'Third-party review':stats_p6['Third-party review'],
           'Recall flagged':stats_p6['Recall rate'],
           'NCT-linked':stats_p6['NCT-linked'],
           'Breakthrough':stats_p6['Breakthrough rate'],
           'PCCP authorized':stats_p6['PCCP adoption']}
fig6b,ax6b=plt.subplots(figsize=(8,4))
bars6b=ax6b.barh(list(h3_flags.keys()),list(h3_flags.values()),color='steelblue',height=0.55)
for bar,rate in zip(bars6b,h3_flags.values()):
    ax6b.text(bar.get_width()+0.005,bar.get_y()+bar.get_height()/2,
              f'{rate:.1%}',va='center',fontsize=9)
ax6b.set_xlim(0,1.15); ax6b.set_xlabel('% of devices')
ax6b.set_title('H3 regulatory flag frequencies')
ax6b.invert_yaxis()
plt.tight_layout()
fig6b.savefig(f'{OUT_SUPP}/Fig6B_H3_flags.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 6B saved.')


Fig 6B saved.


### Fig 6C — "Check-engine light" conceptual framework for algorithmic drift

In [121]:
import matplotlib.patches as mpatches

fig6c, ax6c = plt.subplots(figsize=(13, 7))
ax6c.set_xlim(0, 13); ax6c.set_ylim(0, 7)
ax6c.axis('off')

def fbox(ax, x, y, w, h, color, text, fontsize=9, text_color='white', radius=0.3):
    rect = mpatches.FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle=f'round,pad={radius}',
        facecolor=color, edgecolor='white', linewidth=1.5, zorder=3)
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            color=text_color, fontweight='bold', zorder=4, multialignment='center')

def farrow(ax, x0, y0, x1, y1, color='#555', lw=1.8):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw), zorder=2)

# Title
ax6c.text(6.5, 6.6, '"Check-Engine Light" Framework for Adaptive AI Monitoring',
          ha='center', fontsize=13, fontweight='bold', color='#2c3e50')

# Input signals
signals = [
    (1.4, 5.4, 'Demographic drift\n(E1–E3 shift)'),
    (1.4, 4.1, 'Scanner change\n(F1/F2 update)'),
    (1.4, 2.8, 'New site added\n(D3 expand)'),
    (1.4, 1.5, 'Performance drop\n(post-market)'),
]
for x, y, txt in signals:
    fbox(ax6c, x, y, 2.4, 0.9, '#2980b9', txt, fontsize=8)
ax6c.text(1.4, 6.15, 'Drift triggers', ha='center',
          fontsize=10, color='#2980b9', fontweight='bold')

# Central engine
fbox(ax6c, 5.0, 3.6, 2.6, 1.2, '#c0392b', 'PCCP\nAdaptation Engine\n(H2)', fontsize=10)
for x, y, _ in signals:
    farrow(ax6c, x + 1.2, y, 3.7, 3.6, color='#7f8c8d')

# Risk classification
fbox(ax6c, 8.1, 3.6, 2.4, 1.2, '#e67e22', 'Risk re-score\n(J4)', fontsize=9)
farrow(ax6c, 6.3, 3.6, 6.9, 3.6)

# Response pathways
responses = [
    (11.2, 5.1, '#27ae60', 'LOW RISK\nDocument & monitor'),
    (11.2, 3.6, '#e67e22', 'MODERATE RISK\nTrigger re-validation'),
    (11.2, 2.1, '#c0392b', 'HIGH RISK\nFDA notification\n+ recall review'),
]
for x, y, col, txt in responses:
    fbox(ax6c, x, y, 2.8, 0.95, col, txt, fontsize=8)
    farrow(ax6c, 9.3, 3.6, 9.8, y, color='#555')
ax6c.text(11.2, 6.15, 'Response actions', ha='center',
          fontsize=10, color='#2c3e50', fontweight='bold')

# Feedback loop annotation
ax6c.annotate('', xy=(5.0, 2.2), xytext=(5.0, 3.0),
              arrowprops=dict(arrowstyle='<-', color='#8e44ad', lw=1.5))
ax6c.text(5.35, 2.55, 'Feedback\nloop', fontsize=7.5, color='#8e44ad', va='center')

# Footer
ax6c.text(6.5, 0.35,
    'FDA fields: H2 (PCCP), J4 (risk), A3 (year), F1/F2 (scanner), D3 (sites), E1–E3 (demographics)',
    ha='center', fontsize=8, color='#7f8c8d', style='italic')

plt.tight_layout()
fig6c.savefig(f'{OUT_SUPP}/Fig6C_check_engine_framework.png',
              dpi=150, bbox_inches='tight')
plt.show()
print('Fig 6C saved to OUT_SUPP.')

Fig 6C saved to OUT_SUPP.


---
## Pillar 7 — Strategic Investment Roadmap


In [122]:
def _missing(col):
    return df[col].apply(lambda x: not _reported(x)) if col in df.columns\
           else pd.Series([True]*total_devices)

gap_fields={'Training demographics':'C2a-Training_Demographics_Reported',
            'Training age':'C2b-Training_Age_Distribution',
            'Training sex':'C2c-Training_Sex_Distribution',
            'Training race':'C2d-Training_Race_Ethnicity',
            'Val age':'E1-Val_Age','E2 Val sex':'E2-Val_Sex',
            'Val race':'E3-Val_Race','E4 Val BMI':'E4-Val_BMI',
            'Val prevalence':'E5-Val_Disease_Prevalence',
            'Scanner manufacturers':'F1-Scanner_Manufacturers'}
gap_rates={lbl:_missing(col).sum()/total_devices for lbl,col in gap_fields.items()}
sorted_gaps=sorted(gap_rates.items(),key=lambda it:it[1],reverse=True)

valid_j=df['J4_Composite_Score'].dropna()
stats_p7={'Aggregate generalizability deficit':(15-valid_j).mean() if len(valid_j)>0 else float('nan'),
           'Mean J4 composite score':valid_j.mean(),'SD J4 composite score':valid_j.std()}
for rank,(lbl,rate) in enumerate(sorted_gaps[:5],1):
    stats_p7[f'Gap #{rank}: {lbl}']=rate

n_high=(df['J4_Composite_Score']<=5).sum()
n_no_sc=_missing('F1-Scanner_Manufacturers').sum()
n_low_geo=(df['J1_Score']<4).sum()
impact=[('R1 Metadata enrichment',total_devices),
        ('R2 Post-market mandates (high-risk)',n_high),
        ('R3 Scanner interoperability',n_no_sc),
        ('R4 Cross-border diversification',n_low_geo)]

df_p7=pd.DataFrame(list(stats_p7.items()),columns=['Statistic','Value'])
df_impact=pd.DataFrame(impact,columns=['Recommendation','Devices Impacted'])
with pd.ExcelWriter(f'{OUT_SUPP}/P7_investment_roadmap.xlsx') as w:
    df_p7.to_excel(w,sheet_name='Metrics',index=False)
    df_impact.to_excel(w,sheet_name='Impact_matrix',index=False)
    pd.DataFrame(sorted_gaps,columns=['Field','Non-reporting rate']).to_excel(
        w,sheet_name='All_gap_rates',index=False)
print('P7 core done.'); df_p7


P7 core done.


,Statistic,Value
0,Aggregate generalizability deficit,12.033058
1,Mean J4 composite score,2.966942
2,SD J4 composite score,3.144616
3,Gap #1: Training race,0.943067
4,Gap #2: E4 Val BMI,0.916437
5,Gap #3: Val race,0.898072
6,Gap #4: Val prevalence,0.887052
7,Gap #5: Training sex,0.792470


### 7.2 Gap-severity score per pillar

In [123]:
pillar_gap_scores={
    'P1 Brittleness': float(np.mean([gap_rates.get(f,1) for f in
        ['Training age','Training sex','Training race','Val age','Val race']])),
    'P2 Scanner':      1-stats_p2.get('Multi-vendor rate (J3>=3)',0),
    'P3 Validation':   stats_p3.get('Single-site rate',0),
    'P4 Cross-border': 1-stats_p4.get('Cross-border validation',0),
    'P5 Workforce':    stats_p5.get('HQ Gini',0),
    'P6 Post-market':  1-stats_p6.get('PCCP adoption',0),
    'P7 Synthesis':    stats_p7.get('Aggregate generalizability deficit',0)/15,
}
for p,v in pillar_gap_scores.items(): print(f'  {p}: {v:.3f}')


  P1 Brittleness: 0.830
  P2 Scanner: 0.732
  P3 Validation: 0.022
  P4 Cross-border: 0.908
  P5 Workforce: 0.660
  P6 Post-market: 0.962
  P7 Synthesis: 0.802


### 7.3 Sensitivity analysis — reweighting J1/J2/J3

In [124]:
weight_scenarios={'Equal(1/1/1)':(1,1,1),'Geo-heavy(2/1/1)':(2,1,1),
                  'Demo-heavy(1/2/1)':(1,2,1),'Scanner-heavy(1/1/2)':(1,1,2),'Geo+Demo(2/2/1)':(2,2,1)}
sens=[]
for sc,(w1,w2,w3) in weight_scenarios.items():
    W=w1+w2+w3
    wj=(df['J1_Score']*w1+df['J2_Score']*w2+df['J3_Score']*w3).dropna()
    ms=5*w1+5*w2+5*w3
    deficit=ms-wj.mean()
    phr=((wj/ms*15)<=5).mean()
    sens.append({'Scenario':sc,'Weights':f'{w1}/{w2}/{w3}',
                 'Weighted deficit':round(deficit,2),'% High risk (scaled)':f'{phr:.1%}'})
df_sens=pd.DataFrame(sens)
df_sens.to_excel(f'{OUT_SUPP}/P7_sensitivity_analysis.xlsx',index=False)
print(df_sens.to_string(index=False))


            Scenario Weights  Weighted deficit % High risk (scaled)
        Equal(1/1/1)   1/1/1             12.03                78.1%
    Geo-heavy(2/1/1)   2/1/1             16.26                75.3%
   Demo-heavy(1/2/1)   1/2/1             16.50                78.9%
Scanner-heavy(1/1/2)   1/1/2             15.37                71.4%
     Geo+Demo(2/2/1)   2/2/1             20.73                78.7%


### 7.4 Counterfactual modeling — mandatory reporting since 2020

In [125]:
post2020=df[df['Published_Year']>=2020]; n20=len(post2020)
cf=[]
for lbl,col in gap_fields.items():
    if col not in post2020.columns: continue
    miss=int(_missing(col)[post2020.index].sum())
    cf.append({'Field':lbl,'N since 2020':n20,'Currently missing':miss,
               'Missing %':f'{miss/n20:.1%}','Would report under mandate':miss})
df_cf=pd.DataFrame(cf)
df_cf.to_excel(f'{OUT_SUPP}/P7_counterfactual_2020.xlsx',index=False)
print(f'Counterfactual (N={n20:,} devices since 2020):')
print(df_cf.to_string(index=False))


Counterfactual (N=960 devices since 2020):
                Field  N since 2020  Currently missing Missing %  Would report under mandate
Training demographics           960                646     67.3%                         646
         Training age           960                694     72.3%                         694
         Training sex           960                737     76.8%                         737
        Training race           960                899     93.6%                         899
              Val age           960                709     73.9%                         709
           E2 Val sex           960                673     70.1%                         673
             Val race           960                850     88.5%                         850
           E4 Val BMI           960                872     90.8%                         872
       Val prevalence           960                850     88.5%                         850
Scanner manufacturers      

### Fig 7A — Radar: 7-pillar gap severity

In [126]:
labels7=list(pillar_gap_scores.keys())
vals7=list(pillar_gap_scores.values())
N7=len(labels7)
ang7=[n/float(N7)*2*math.pi for n in range(N7)]
ang7+=ang7[:1]; vals7+=vals7[:1]
fig7a,ax7a=plt.subplots(figsize=(6,6),subplot_kw=dict(polar=True))
ax7a.set_theta_offset(math.pi/2); ax7a.set_theta_direction(-1)
ax7a.set_xticks(ang7[:-1]); ax7a.set_xticklabels(labels7,size=9)
ax7a.set_ylim(0,1)
ax7a.plot(ang7,vals7,color='tomato',linewidth=2)
ax7a.fill(ang7,vals7,color='tomato',alpha=0.25)
ax7a.set_title('7-pillar gap severity (higher = worse)',pad=20)
plt.tight_layout()
fig7a.savefig(f'{OUT_MAIN}/Fig7A_radar_gap_severity.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 7A saved.')


Fig 7A saved.


### Fig 7B — Waterfall: investment allocation

In [127]:
inv_lbls=['Metadata\nEnrichment','Demographic\nReporting','Scanner\nInterop',
          'Multi-site\nValidation','PCCP\nExpansion']
inv_vals=[int(_missing('C1a-Training_Data_Described').sum()),
          int(_missing('E3-Val_Race').sum()),
          int(n_no_sc),
          int(stats_p3.get('Single-site rate',0)*total_devices),
          int((1-stats_p6.get('PCCP adoption',0))*total_devices)]
fig7b,ax7b=plt.subplots(figsize=(9,5))
cols7b=['#2980b9','#8e44ad','#e67e22','#27ae60','#c0392b']
bars7b=ax7b.bar(inv_lbls,inv_vals,color=cols7b,edgecolor='white',width=0.6)
for bar,val in zip(bars7b,inv_vals):
    ax7b.text(bar.get_x()+bar.get_width()/2,bar.get_height()+5,
              f'{val:,}',ha='center',va='bottom',fontsize=9,fontweight='bold')
ax7b.set_ylabel('Number of devices impacted')
ax7b.set_title('Estimated devices impacted per investment lever')
plt.tight_layout()
fig7b.savefig(f'{OUT_SUPP}/Fig7B_investment_allocation.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 7B saved.')


Fig 7B saved.


### Fig 7C — Forest plot: reporting rates with 95% CIs

In [140]:
f_lbls=list(reporting_rates_for_fig.keys())
f_rates=[reporting_rates_for_fig[l] for l in f_lbls]
f_lo=[ci_dict_for_fig[l][0] for l in f_lbls]
f_hi=[ci_dict_for_fig[l][1] for l in f_lbls]
oi=sorted(range(len(f_rates)),key=lambda i:f_rates[i], reverse=True)
f_lbls=[f_lbls[i] for i in oi]; f_rates=[f_rates[i] for i in oi]
f_lo=[f_lo[i] for i in oi]; f_hi=[f_hi[i] for i in oi]
y_pos=range(len(f_lbls))
xerr_lo=[f_rates[i]-f_lo[i] for i in range(len(f_lbls))]
xerr_hi=[f_hi[i]-f_rates[i] for i in range(len(f_lbls))]
fig7c,ax7c=plt.subplots(figsize=(8,max(4,len(f_lbls)*0.55)))
ax7c.errorbar(f_rates,list(y_pos),xerr=[xerr_lo,xerr_hi],
              fmt='o',color='steelblue',ecolor='steelblue',capsize=4,markersize=6,linewidth=1.4)
ax7c.set_yticks(list(y_pos)); ax7c.set_yticklabels(f_lbls,fontsize=9)
ax7c.set_xlim(0,1.1)
ax7c.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_:f'{x:.0%}'))
ax7c.set_xlabel('Reporting rate (95% Wilson CI)')
ax7c.set_title('Reporting rates forest plot (sorted by severity)')
ax7c.axvline(0.5,color='tomato',linestyle='--',linewidth=1,alpha=0.6)
for i,(rate,lo,hi) in enumerate(zip(f_rates,f_lo,f_hi)):
    ax7c.text(hi+0.02,i,f'{rate:.0%}',va='center',fontsize=8)
plt.tight_layout()
fig7c.savefig(f'{OUT_SUPP}/Fig7C_forest_plot.png',dpi=150,bbox_inches='tight')
plt.show(); print('Fig 7C saved.')


Fig 7C saved.


### Table 7 — Recommendations table

In [129]:
table7=pd.DataFrame([
    {'Recommendation':'Mandate training data disclosure (C1a-C1d)',
     'Evidence base':'P1 training desc rate; P4 HHI USA dominance',
     'Implementation barrier':'Political: industry opposition; Technical: low'},
    {'Recommendation':'Require demographic reporting for validation (E1-E3)',
     'Evidence base':'P1 reporting cliff; Cochran-Armitage trend',
     'Implementation barrier':'Moderate: EHR linkage; privacy concerns'},
    {'Recommendation':'Mandate >=2 scanner manufacturers for CT/MRI (F1/J3)',
     'Evidence base':'P2 multi-vendor rate; Kruskal-Wallis',
     'Implementation barrier':'Moderate: scanner access; cost'},
    {'Recommendation':'Require multi-site (>=4) validation for high-risk AI (D2)',
     'Evidence base':'P3 single-site rate; logistic regression',
     'Implementation barrier':'High: site recruitment; cost; time'},
    {'Recommendation':'Expand PCCP + link to post-market demographic surveillance (H2)',
     'Evidence base':'P6 PCCP rate; logistic regression year trend',
     'Implementation barrier':'Low: guidance exists; adoption lags'},
])
table7.to_excel(f'{OUT_SUPP}/Table7_recommendations.xlsx',index=False)
print('Table 7 saved.'); table7


Table 7 saved.


,Recommendation,Evidence base,Implementation barrier
0,Mandate training data disclosure (C1a-C1d),P1 training desc rate; P4 HHI USA dominance,Political: industry opposition; Technical: low
1,Require demographic reporting for validation (...,P1 reporting cliff; Cochran-Armitage trend,Moderate: EHR linkage; privacy concerns
2,Mandate >=2 scanner manufacturers for CT/MRI (...,P2 multi-vendor rate; Kruskal-Wallis,Moderate: scanner access; cost
3,Require multi-site (>=4) validation for high-r...,P3 single-site rate; logistic regression,High: site recruitment; cost; time
4,Expand PCCP + link to post-market demographic ...,P6 PCCP rate; logistic regression year trend,Low: guidance exists; adoption lags


---
## Cross-Cutting Analytical Toolkit


In [130]:
# HHI summary
t_hhi=herfindahl_hirschman_index(pd.Series([c for lst in df['Train_Countries_Set'] for c in lst]).value_counts().values)
v_hhi=herfindahl_hirschman_index(pd.Series([c for lst in df['Val_Countries_Set'] for c in lst]).value_counts().values)
print(f'Training-country HHI:   {t_hhi:.0f}')
print(f'Validation-country HHI: {v_hhi:.0f}')
print(f'Manufacturer HHI:       {stats_p2.get("Manufacturer HHI",float("nan")):.0f}')


Training-country HHI:   2964
Validation-country HHI: 3739
Manufacturer HHI:       2144


In [131]:
# Ordinal regression: J4 risk class
ord_j4=fit_ordinal_regression(df.dropna(subset=['J4_Risk_Level']),'J4_Risk_Level',
    ['High risk','Moderate risk','Low risk'],
    ['Published_Year','Pathway','Primary_Modality'])
print(ord_j4.summary())


                             OrderedModel Results                             
Dep. Variable:          J4_Risk_Level   Log-Likelihood:                -590.15
Model:                   OrderedModel   AIC:                             1214.
Method:            Maximum Likelihood   BIC:                             1299.
Date:                Tue, 26 May 2026                                         
Time:                        16:51:34                                         
No. Observations:                1088                                         
Df Residuals:                    1071                                         
Df Model:                          15                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Pathway_De Novo                  1.4881        nan        nan        nan         nan         na

In [132]:
# Chi-square Bonferroni across modalities
alpha_b=0.05/len(DEMO_FIELDS)
print(f'Bonferroni alpha: {alpha_b:.5f}')
for lbl,col in DEMO_FIELDS.items():
    if col not in df.columns: continue
    rep=df[col].apply(_reported).astype(int)
    ct=pd.crosstab(df['Primary_Modality'],rep)
    if ct.shape[1]==2 and len(ct)>=2:
        stat,p,_,_=chi2_contingency(ct)
        sig='(*)' if p<alpha_b else '(ns)'
        print(f'{lbl:25s}  chi2={stat:.2f}  p={p:.4f}  {sig}')


Bonferroni alpha: 0.00625
Training Age               chi2=36.80  p=0.0001  (*)
Training Sex               chi2=40.09  p=0.0000  (*)
Training Race              chi2=16.69  p=0.1175  (ns)
Val Age                    chi2=25.43  p=0.0079  (ns)
Val Sex                    chi2=50.67  p=0.0000  (*)
Val Race                   chi2=32.44  p=0.0006  (*)
Val BMI                    chi2=163.52  p=0.0000  (*)
Val Prevalence             chi2=125.10  p=0.0000  (*)


In [133]:
# Predicate chain top-100 summary
print('Top-100 most recent devices:')
print(f'  Predicate age: mean={pa100.mean():.1f} median={pa100.median():.1f} max={pa100.max():.0f} yrs')
print(f'  Chain depth:   mean={pd100.mean():.1f} max={pd100.max():.0f} generations')


Top-100 most recent devices:
  Predicate age: mean=2.9 median=1.5 max=13 yrs
  Chain depth:   mean=2.0 max=6 generations


## Final output summary

In [134]:
print('\n=== FDA 7-Pillar Audit - All analyses complete ===')
print(f'Cohort: {total_devices:,} devices')
print(f'Supplementary: {OUT_SUPP}')
print(f'Main figures:  {OUT_MAIN}')
print('\nFigures generated:')
for f in sorted(os.listdir(OUT_MAIN)): print(f'  {f}')



=== FDA 7-Pillar Audit - All analyses complete ===
Cohort: 1,089 devices
Supplementary: ../output/supplementary
Main figures:  ../output/main_exhibits

Figures generated:
  Fig1A_reporting_cliff.png
  Fig1B_J4_risk_by_pathway.png
  Fig3B_predicate_age_histogram.png
  Fig3E_validation_choropleth.png
  Fig4A_sankey_HQ_train_val.png
  Fig7A_radar_gap_severity.png
